In [19]:
from platform import python_version

print(python_version())

3.9.25


In [1]:
import requests
import numpy as np
import pandas as pd
from datetime import datetime, timezone, timedelta
import time as time_utils
from astropy.time import Time
from astropy.coordinates import SkyCoord
from astropy import units as u
import json
from itertools import chain
from astropy.io import fits
from astropy.time import Time
import matplotlib.pyplot as plt
from astropy.wcs import WCS
from astropy.coordinates import EarthLocation, AltAz, SkyOffsetFrame
import sys
import os
print(os.path.exists(r"C:\Users\carlo\RMS"))
print(os.path.exists(r"C:\Users\carlo\RMS\RMS"))  # the inner RMS package folder
sys.path.append(r"C:\Users\carlo\RMS")
from RMS.Formats.Platepar import Platepar
from RMS.Astrometry.ApplyAstrometry import raDecToXYPP, xyToRaDecPP
from photutils.centroids import centroid_2dg, centroid_sources
from photutils.aperture import CircularAperture, aperture_photometry
from RMS.Astrometry.Conversions import jd2Date
from skimage.draw import polygon
from photutils.background import Background2D, MedianBackground
from photutils.centroids import centroid_com, centroid_2dg, centroid_sources
from photutils.aperture import CircularAperture, CircularAnnulus, aperture_photometry, RectangularAperture, RectangularAnnulus
from photutils.profiles import RadialProfile, CurveOfGrowth
from astropy.stats import sigma_clipped_stats
from numpy import log10
import subprocess
import json
from numpy import sqrt, cos
from astroquery.simbad import Simbad
import astroalign as aa
from skimage.transform import estimate_transform
from astropy.time import TimeDelta
from skimage.transform import AffineTransform, SimilarityTransform
from skimage.measure import ransac
import warnings
from astropy.utils.exceptions import AstropyUserWarning
from scipy.interpolate import splprep, splev
from scipy.ndimage import map_coordinates
from scipy.optimize import curve_fit
from scipy.spatial import cKDTree
from astropy.stats import SigmaClip
# Ignore Photutils centroid warnings
with warnings.catch_warnings():
    warnings.simplefilter('ignore', AstropyUserWarning)
import cv2
import logging
logging.getLogger('matplotlib').setLevel(logging.WARNING)
from skimage.transform import radon
from RMS.Astrometry.Conversions import date2JD
from scipy.ndimage import map_coordinates
from scipy.optimize import curve_fit
from skimage.measure import ransac
from scipy.special import erf
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import FormatStrFormatter
from photutils.segmentation import detect_sources
from photutils.utils import circular_footprint
from photutils.psf import EPSFBuilder
from astropy.table import Table
from astropy.nddata import NDData
from photutils.psf import extract_stars, EPSFBuilder
from scipy.interpolate import RectBivariateSpline
from astropy.utils.exceptions import AstropyUserWarning
import numpy as np
from scipy.interpolate import CubicSpline
from scipy.integrate import simpson
from scipy.constants import h, c, k as k_B
warnings.filterwarnings('ignore', category=AstropyUserWarning)
from photutils.utils.exceptions import NoDetectionsWarning
warnings.simplefilter('ignore', category=NoDetectionsWarning)
import photutils as pt
print(pt.__version__)
import skimage as sk
print(sk.__version__)
import scipy as sp
print(sp.__version__)
print(np.__version__)
import astropy as ap
print(ap.__version__)
print(cv2.__version__)
from astropy.modeling import models, fitting
from scipy.stats import t as student_t
from astropy.modeling import Fittable2DModel, Parameter

True
True
Camera settings file: C:\Users\carlo\RMS\camera_settings.json
Disabled upload because the default station code is used!


No `name` configuration, performing automatic discovery
running build_ext
skipping 'RMS.Astrometry.CyFunctions' extension (up-to-date)
Popen(['git', 'version'], cwd=C:\Users\carlo\Summer Project, stdin=None, shell=False, universal_newlines=False)
Popen(['git', 'version'], cwd=C:\Users\carlo\Summer Project, stdin=None, shell=False, universal_newlines=False)


1.11.0
0.24.0
1.13.1
1.26.4
6.0.1
4.11.0


In [25]:
class PolyModel:
    """
    Defined as a seperate class for my curve fitting procedure.
    """
    def __init__(self, order=2):
        self.order = order
        self.coeffs = None

    def estimate(self, data):
        x, y = data[:, 0], data[:, 1]
        self.coeffs = np.polyfit(x, y, self.order)
        return True

    def residuals(self, data):
        x, y = data[:, 0], data[:, 1]
        y_pred = np.polyval(self.coeffs, x)
        return np.abs(y - y_pred)

    def predict(self, x):
        return np.polyval(self.coeffs, x)

class EllipticalMoffat2D(Fittable2DModel):
    amplitude = Parameter(default=1)
    x_0 = Parameter(default=0)
    y_0 = Parameter(default=0)
    gamma_a = Parameter(default=1)
    gamma_b = Parameter(default=1)
    theta = Parameter(default=0)
    alpha = Parameter(default=3.5, fixed=True)

    @staticmethod
    def evaluate(x, y, amplitude, x_0, y_0, gamma_a, gamma_b, theta, alpha):
        c, s = np.cos(theta), np.sin(theta)
        xr = (x - x_0)*c + (y - y_0)*s
        yr = -(x - x_0)*s + (y - y_0)*c
        return amplitude * (1 + (xr/gamma_a)**2 + (yr/gamma_b)**2)**(-alpha)
        
def make_poly_model(order):
    class _BoundPolyModel(PolyModel):
        def __init__(self):
            super().__init__(order=order)
    return _BoundPolyModel

class OperatingFitsFiles:
    """
    I like my classes, they are easy to work with
    """
    def __init__(self, file_name, site, simulated):
        self.file_name = file_name
        self.site = site
        self.median_fwhm  = 10
        self.simulated = simulated
        
    def opening_fits_file(self):
        with fits.open(self.file_name, memmap=False) as hdul:
            header = hdul[0].header
            self.header = header
            print(header)
            exposure_start = Time(header['DATE-OBS'], scale = 'utc') 
            
            self.exposure_start_jd = exposure_start.jd 
            
            print(self.exposure_start_jd)

            self.exposure_start = Time(header["DATE-OBS"], format="isot", scale="utc")
            
            self.exposure_start_dt = datetime.fromisoformat(f"{exposure_start}").replace(tzinfo=timezone.utc)
    
            self.exposure_time = header.get('EXPTIME', 0)
            
            self.exposure_end_dt = self.exposure_start_dt + timedelta(seconds = self.exposure_time)
            
            self.right_ascension = header.get('RA', 0)
            self.declination = header.get('DEC', 0)
            
            self.ra = header['RA'] # Not sure why I did this twice, but will clear up post project!
            self.dec = header['DEC']
            print(self.ra, self.dec)
            self.lon = header['HIERARCH ESO TEL GEOLON']
            self.lat = header['HIERARCH ESO TEL GEOLAT']
            self.alt = header['HIERARCH ESO TEL GEOELEV']
            self.location = EarthLocation(lat=self.lat*u.deg, lon=self.lon*u.deg, height=self.alt*u.m)
            if self.simulated:
                self.ra_truth_start = header['TRUERA0']
                self.dec_truth_start = header['TRUEDEC0']
                self.ra_truth_end = header['TRUERA1']
                self.dec_truth_end = header['TRUEDEC1']

        self.fits_data = np.flipud(fits.getdata(self.file_name))
        
        box_size = 6400
        ny, nx = self.fits_data.shape
        half = box_size // 2
        cy, cx = ny // 2, nx // 2  # center pixel
        # Compute crop boundaries
        y0, y1 = cy - half, cy + half
        x0, x1 = cx - half, cx + half
        
        cropped = self.fits_data[y0:y1, x0:x1]
        """
        sigma_clip = SigmaClip(sigma=1.0)
        bkg_estimator = MedianBackground()
            
        local_bkg = Background2D(
                    cropped,
                    box_size=(32, 32),
                    filter_size=(3, 3),
                    sigma_clip=sigma_clip,
                    bkg_estimator=bkg_estimator,
                    exclude_percentile=100,  
        )
        
        cropped = cropped - local_bkg.background
        """
        cropped = np.clip(cropped, a_min=None, a_max=1500)
        
        # radial mask relative to the cropped array's own center
        hh, ww = cropped.shape
        yy, xx = np.mgrid[0:hh, 0:ww]
        ccy, ccx = hh // 2, ww // 2
        radius = half  # inscribed circle
        dist = np.sqrt((yy - ccy) ** 2 + (xx - ccx) ** 2)
        mask = dist <= radius
        
        fill_value = np.median(cropped[mask])  # avoid a hard 0-edge that Canny would pick up
        cropped[~mask] = fill_value
        
        self.fits_data_cropped = cropped
        
        star_names = "Spica" 

        star_location = SkyCoord.from_name(star_names)
        
        pp = Platepar()
        pp.read(r"C:\Users\carlo\Summer Project\newlenscalib.cal")
        star_x, star_y = raDecToXYPP(np.atleast_1d(star_location.ra.deg), np.atleast_1d(star_location.dec.deg), self.exposure_start_jd, pp)
        
        centroid_func = centroid_2dg
        print(star_x, star_y)
        x_ref, y_ref = centroid_sources(
                self.fits_data, star_x, star_y, box_size=39, centroid_func=centroid_func
            )

        if np.sqrt((star_x - x_ref)**2 +(star_y - y_ref)**2) < 10:
            self.pp = Platepar()
            self.pp.read(r"C:\Users\carlo\Summer Project\newlenscalib.cal")
        else:
            self.pp = Platepar()
            self.pp.read(r"C:\Users\carlo\Summer Project\lenscalibMAIN.cal")
        
        

        print(self.pp)

        
    def satellites_overhead(self):
        visible_satellites = []

        response = requests.get(
            f"https://satchecker.cps.iau.org/fov/satellites-above-horizon/?site={self.site}&julian_date={self.exposure_start_jd}&min_altitude=20&illimunated_only=true",
            timeout = 60
            )
        

    def vignetting_response(self, flux, x, y):
        """
        Corrects the vignetting of any selected pixel within the image field.
        """
        vignetting_coefficient = 0.000204
        vignetting_coefficient_error = 0.000015
        radius = sqrt((float(x) - 4375)**2 + (float(y)- 4375)**2)
        
        vignetted_intensity = flux / (cos(vignetting_coefficient * radius)**4)
        
        vignetted_intensity_error_lb = flux / (cos((vignetting_coefficient + vignetting_coefficient_error) * radius)**4)
        vignetted_intensity_error_ub = flux / (cos((vignetting_coefficient - vignetting_coefficient_error) * radius)**4)

        error_lb = (vignetted_intensity - vignetted_intensity_error_lb) / vignetted_intensity * 100
        error_ub = (vignetted_intensity - vignetted_intensity_error_ub) / vignetted_intensity * 100
        
        return vignetted_intensity, error_lb, error_ub



    def fit_moffat_gamma(self, x, y, box_size=19, beta=2.5):
        h = box_size // 2
        xi, yi = int(round(x)), int(round(y))
        cut = self.fits_data[yi-h:yi+h+1, xi-h:xi+h+1].astype(float)
        if cut.shape != (box_size, box_size):
            return None
    
        _, sky, _ = sigma_clipped_stats(cut)
        cut = cut - sky
    
        yy, xx = np.mgrid[:box_size, :box_size]
        init = models.Moffat2D(amplitude=cut.max(), x_0=h, y_0=h,
                               gamma=2.0, alpha=beta)
        init.alpha.fixed = True          # astropy 'alpha' == Moffat index beta
    
        fitter = fitting.LevMarLSQFitter()
        m = fitter(init, xx, yy, cut)
    
        if fitter.fit_info['ierr'] not in (1, 2, 3, 4):
            return None
        return float(abs(m.gamma.value))     # core width, px    
    def photometry_analysis(self, x, y, source_radius=None,
                        annulus_inner=None, annulus_outer=None):
        print(source_radius)
        if source_radius is None:
            source_radius = 1.5 * self.median_fwhm
        if annulus_inner is None:
            annulus_inner = 4.0 * self.median_fwhm
        if annulus_outer is None:
            annulus_outer = annulus_inner + 4.0 * self.median_fwhm
    
        central_aperture = CircularAperture((x, y), source_radius)
        annulus_aperture = CircularAnnulus((x, y), annulus_inner, annulus_outer)
    
        cen_pho = aperture_photometry(self.fits_data, central_aperture)
    
        annulus_mask = annulus_aperture.to_mask(method='center')
        annulus_data = annulus_mask.multiply(self.fits_data)
        annulus_data_1d = annulus_data[annulus_mask.data > 0]
        _, bkg_median, _ = sigma_clipped_stats(annulus_data_1d)
    
        bkg_total_in_source_aperture = bkg_median * central_aperture.area
        return cen_pho['aperture_sum'][0] - bkg_total_in_source_aperture


    def return_data(self):
        return self.data
    
    def streak_passes(self, ra, dec, radius):
        """
        I should really start adding stuff here!!!
        """
        satellites = {}
    
        url = 'https://satchecker.cps.iau.org/fov/satellite-passes/'
        duration = 120
    
        params = {'site': 'paranal',
                  'start_time_jd': self.exposure_start_jd,
                  'duration': duration,
                  'ra': ra,
                  'dec': dec,
                  'fov_radius': radius,
                  'group_by': 'satellite',
                  'async': False}

        # The addition of these guards were implemented with the help of Claude Opus (Sped up the process of writing myself, lazy I know)
        try:
            r = requests.get(url, params=params, timeout=30)
        except requests.exceptions.RequestException as e:
            print(f"streak_passes: request failed ({e}) for RA={ra:.4f}, Dec={dec:.4f}")
            return satellites
    
        if r.status_code != 200:
            print(f"streak_passes: HTTP {r.status_code} for RA={ra:.4f}, Dec={dec:.4f} — {r.text[:200]}")
            return satellites
    
        try:
            data = r.json()
        except ValueError:
            print(f"streak_passes: response wasn't valid JSON for RA={ra:.4f}, Dec={dec:.4f}")
            return satellites
    
        if not isinstance(data, dict) or 'data' not in data or 'satellites' not in data.get('data', {}):
            print(f"streak_passes: no satellite data in response for RA={ra:.4f}, Dec={dec:.4f} — {data}")
            return satellites
    
        sat_block = data['data']['satellites']
        if not sat_block:
            print(f"streak_passes: no satellites found within radius={radius} of RA={ra:.4f}, Dec={dec:.4f}")
            return satellites
    
        for sat_key, sat_data in sat_block.items():
    
            if sat_key not in satellites:
                satellites[sat_key] = []
    
            for position in sat_data['positions']:
    
                x, y = raDecToXYPP(np.atleast_1d(position['ra']), np.atleast_1d(position['dec']),
                                    position['julian_date'], self.pp)
                satellites[sat_key].append([
                    x,
                    y,
                    position['ra'],
                    position['dec'],
                    position['julian_date'],
                    position['range_km'],
                    sat_data['norad_id']
                ])
    
        return satellites

    from scipy.interpolate import RectBivariateSpline

    @staticmethod
    def kasten_formula(zenith_angle):
        return (np.cos(np.radians(zenith_angle)) + 0.50572*(96.07995 - zenith_angle)**(-1.6364))**(-1)

    # Pecaut & Mamajek (2013), ApJS 208, 9 -- representative dwarf sequence anchor points   
    @staticmethod
    def bv_to_teff(bv, _PM_BV, _PM_TEFF):
        bv_clamped = np.clip(bv, _PM_BV.min(), _PM_BV.max())
        return np.interp(bv_clamped, _PM_BV, _PM_TEFF)

    def extinction_magnitude_correction(self, zenith_angle, vmag, bmag):
        # Paranal extinction values obtained by Patat et al. 2011
        # https://www.aanda.org/articles/aa/full_html/2011/03/aa15537-10/aa15537-10.html
        # We note that between 6775-7000 we have interpolated.
        _PM_BV   = np.array([-0.33, -0.30, -0.24, -0.17, -0.11, -0.02, 0.00, 0.15,
                           0.30,  0.44,  0.58,  0.68,  0.82,  0.92,  1.15, 1.40, 1.80])
        _PM_TEFF = np.array([42000, 30000, 20000, 15700, 12500, 10000, 9700, 8200,
                           7200,  6650,  6050,  5770,  5250,  4900,  4350, 3800, 3200])
        
        wavelength_A = np.array([
            4025, 4075, 4125, 4175, 4225, 4275, 4325, 4375, 4425, 4475,
            4525, 4575, 4625, 4675, 4725, 4775, 4825, 4875, 4925, 4975,
            5025, 5075, 5125, 5175, 5225, 5275, 5325, 5375, 5425, 5475,
            5525, 5575, 5625, 5675, 5725, 5775, 5825, 5875, 5925, 5975,
            6025, 6075, 6125, 6175, 6225, 6275, 6325, 6375, 6425, 6475,
            6525, 6575, 6625, 6675, 6725, 6775, 7000
        ])
    
        k_lambda = np.array([
            0.330, 0.316, 0.298, 0.285, 0.274, 0.265, 0.253, 0.241, 0.229, 0.221,
            0.212, 0.204, 0.198, 0.190, 0.185, 0.182, 0.176, 0.169, 0.162, 0.157,
            0.156, 0.153, 0.146, 0.143, 0.141, 0.139, 0.139, 0.134, 0.133, 0.131,
            0.129, 0.127, 0.128, 0.130, 0.134, 0.132, 0.124, 0.122, 0.125, 0.122,
            0.117, 0.115, 0.108, 0.104, 0.102, 0.099, 0.095, 0.092, 0.085, 0.086,
            0.083, 0.081, 0.076, 0.072, 0.068, 0.064, 0.064
        ])
    
        sigma_k = np.array([
            0.004, 0.004, 0.004, 0.004, 0.004, 0.004, 0.004, 0.003, 0.003, 0.003,
            0.003, 0.003, 0.003, 0.003, 0.003, 0.003, 0.003, 0.003, 0.003, 0.003,
            0.003, 0.003, 0.003, 0.003, 0.003, 0.003, 0.002, 0.002, 0.002, 0.002,
            0.002, 0.002, 0.002, 0.002, 0.002, 0.002, 0.002, 0.003, 0.003, 0.003,
            0.002, 0.002, 0.002, 0.002, 0.002, 0.002, 0.002, 0.002, 0.002, 0.003,
            0.003, 0.002, 0.002, 0.002, 0.002, 0.002, 0.002
        ])
    
        # QE values obtained for an adjacent sensor (ours does not have QE curve data available!)
        transmission_filter = 0.95
        qe_wavelength_nm = np.array([400, 425, 450, 475, 500, 525, 550, 575, 600, 625, 650, 675, 700])
        qe_percent = np.array([64, 74, 78, 80, 80, 78, 72, 67, 61, 55, 48, 43, 37]) * transmission_filter * 0.01
        qe_wavelength_A = qe_wavelength_nm * 10
    
        lam_min = max(wavelength_A.min(), qe_wavelength_A.min())
        lam_max = min(wavelength_A.max(), qe_wavelength_A.max())
        lam_common = np.arange(lam_min, lam_max + 1, 5)
    
        k_interp     = CubicSpline(wavelength_A, k_lambda)(lam_common)
        sigma_interp = CubicSpline(wavelength_A, sigma_k)(lam_common)
        S_interp     = np.interp(lam_common, qe_wavelength_A, qe_percent)
    
        # B-V colour -> approximate effective temperature (Ballesteros 2012)
        bv = bmag - vmag
        T_eff = self.bv_to_teff(bv, _PM_BV, _PM_TEFF)
        
        lam_m = lam_common * 1e-10
        planck = (2 * h * c**2) / (lam_m**5 * (np.exp((h * c) / (lam_m * k_B * T_eff)) - 1))
        weight = S_interp * planck

        print(f"vmag={vmag:.2f} bmag={bmag:.2f} B-V={bv:.3f} T_eff={T_eff:.1f}")
        numerator   = simpson(k_interp * weight, x=lam_common)
        denominator = simpson(weight, x=lam_common)
        extinction_coeff = numerator / denominator
    
        var_num = simpson((sigma_interp**2) * (S_interp**2), x=lam_common)
        sigma_extinction_coeff = np.sqrt(var_num) / denominator
    
        X = self.kasten_formula(zenith_angle)
        m_correction = X * extinction_coeff
    
        return m_correction

        from scipy.optimize import curve_fit
    from scipy.stats import t as student_t
    
    
    @staticmethod
    def _r_ee(alpha, beta, EE):
        """2D circular Moffat enclosed-energy radius."""
        return alpha * np.sqrt((1 - EE)**(1/(1 - beta)) - 1)
    
    @staticmethod
    def _hw_ee(alpha, beta, EE):
        """Cross-trail enclosed-energy halfwidth (infinite-line limit; valid for L >> alpha)."""
        nu = 2*beta - 2
        return alpha * student_t.ppf(0.5 + EE/2, nu) / np.sqrt(nu)
    
    
    def fit_moffat(self, xc, yc, box=10):
        """Free alpha and beta. Returns (alpha, beta) or None."""
        y0, x0 = int(round(yc)), int(round(xc))
        cut = self.fits_data[y0-box:y0+box+1, x0-box:x0+box+1].astype(float)
        if cut.shape != (2*box+1, 2*box+1):
            return None
        yy, xx = np.mgrid[0:cut.shape[0], 0:cut.shape[1]]
        cx, cy = xc - x0 + box, yc - y0 + box
    
        def model(coords, amp, mx, my, alpha, beta, bkg):
            x, y = coords
            return (amp*(1 + ((x-mx)**2 + (y-my)**2)/alpha**2)**(-beta) + bkg).ravel()
    
        try:
            p, _ = curve_fit(model, (xx, yy), cut.ravel(),
                             p0=[cut.max()-np.median(cut), cx, cy, 3.0, 3.0, np.median(cut)],
                             sigma=np.sqrt(np.maximum(cut, 1)).ravel(),
                             bounds=([0, cx-3, cy-3, 0.5, 1.5, -np.inf],
                                     [np.inf, cx+3, cy+3, 20.0, 15.0, np.inf]),
                             maxfev=20000)
        except (RuntimeError, ValueError):
            return None
        return p[3], p[4]
    
    
    def ee_radius_star(self, EE=0.8):
        """Clipped mean of the per-star EE radius (average the radius, not the parameters)."""
        a, b = self.moffat_fits[:, 0], self.moffat_fits[:, 1]
        _, med, _ = sigma_clipped_stats(self._r_ee(a, b, EE))
        return med
    
    def ee_halfwidth_streak(self, EE=0.80):
        a, b = self.moffat_fits[:, 0], self.moffat_fits[:, 1]
        _, med, _ = sigma_clipped_stats(self._hw_ee(a, b, EE))
        return med
    
    
    def ee_radius_scan(self, stars, radii=None, r_in=None, r_out=None):
        """
        Empirical aperture scan. Minimises the scatter in zeropoint across stars,
        which has a real minimum (unlike normalised EE, which trivially goes to zero
        at the normalisation radius).
    
        stars: list of (xc, yc, vmag, mag_correction).
        Returns (r_best, sd_best_mag, radii, sd_mag).
        """
        a_med = np.median(self.moffat_fits[:, 0])
        if radii is None:
            radii = np.arange(2.0, 8.0*a_med, 0.5)
        if r_in is None:
            r_in = 5.0 * a_med
        if r_out is None:
            r_out = 8.0 * a_med
    
        rows = []
        for xc, yc, vmag, mag_corr in stars:
            f = np.array([self.photometry_analysis(xc, yc, r, r_in, r_out) for r in radii])
            with np.errstate(invalid="ignore", divide="ignore"):
                zp = vmag + 2.5*np.log10(np.where(f > 0, f, np.nan)) + mag_corr
            rows.append(zp)
    
        zp = np.array(rows)
        sd = np.nanstd(zp, axis=0, ddof=1)
        ok = np.isfinite(sd) & (np.sum(np.isfinite(zp), axis=0) >= 3)
        if not ok.any():
            return np.nan, np.nan, radii, sd
        i = np.flatnonzero(ok)[np.nanargmin(sd[ok])]
        return radii[i], sd[i], radii, sd
    
    
    def plot_stars(self, star_centres, r_star, r_in, r_out, box=50):
        import matplotlib.pyplot as plt
        from matplotlib.patches import Circle
    
        n = len(star_centres)
        ncols = 5
        nrows = int(np.ceil(n / ncols))
        fig, axes = plt.subplots(nrows, ncols, figsize=(3*ncols, 3*nrows))
        axes = np.atleast_1d(axes).ravel()
    
        h = box // 2
        for ax, (xc, yc) in zip(axes, star_centres):
            x0, y0 = int(round(xc)) - h, int(round(yc)) - h
            cut = self.fits_data[y0:y0+box, x0:x0+box]
            if cut.shape != (box, box):
                ax.axis("off")
                continue
            vmin, vmax = np.percentile(cut, [5, 99])
            ax.imshow(cut, origin="lower", cmap="gray", vmin=vmin, vmax=vmax,
                      extent=[x0-0.5, x0+box-0.5, y0-0.5, y0+box-0.5])
            for r, c in ((r_star, "lime"), (r_in, "cyan"), (r_out, "cyan")):
                ax.add_patch(Circle((xc, yc), r, fill=False, color=c, lw=1))
            ax.set_title(f"({xc:.0f}, {yc:.0f})", fontsize=8)
            ax.set_xticks([]); ax.set_yticks([])
    
        for ax in axes[n:]:
            ax.axis("off")
        plt.tight_layout()
        plt.show()
    def stellar_calibrations(self, location, n_refs=10, max_sep_px=3000, EE=0.80):
        """
        location: (ra, dec) in degrees for the target star/streak.
        n_refs: max number of nearby calibration stars to use.
        max_sep_px: don't use a calibration star farther than this from the target,
                    even if fewer than n_refs are found.
        EE: enclosed-energy fraction defining the photometric aperture. The streak
            halfwidth must use the same fraction so the aperture correction cancels.
        """
        centroid_func = centroid_2dg
    
        ra_loc, dec_loc = location
        x_loc_arr, y_loc_arr = raDecToXYPP(
            np.atleast_1d(float(ra_loc)), np.atleast_1d(float(dec_loc)),
            self.exposure_start_jd, self.pp
        )
        x_loc, y_loc = x_loc_arr[0], y_loc_arr[0]
    
        candidates = []
        with open("stellar_calibration1.txt") as f:
            for line in f:
                star_id, ra, dec, vmag, bmag, sg = line.strip().split(",")
    
                ra_arr = np.atleast_1d(float(ra))
                dec_arr = np.atleast_1d(float(dec))
                x_pred, y_pred = raDecToXYPP(ra_arr, dec_arr,
                                             self.exposure_start_jd, self.pp)
                x, y = x_pred[0], y_pred[0]
    
                radius = sqrt((x - 4375)**2 + (y - 4375)**2)
                if radius >= 3500:
                    continue
    
                sep = sqrt((x - x_loc)**2 + (y - y_loc)**2)
                if sep > max_sep_px:
                    continue
    
                altaz_frame = AltAz(obstime=self.exposure_start, location=self.location)
                star_coords = SkyCoord(ra=ra_arr, dec=dec_arr, unit=(u.deg, u.deg))
                star_altaz = star_coords.transform_to(altaz_frame)
                zenith_angle = 90 - star_altaz.alt.deg[0]
    
                mag_correction = self.extinction_magnitude_correction(
                    float(zenith_angle), float(vmag), float(bmag)
                )
                candidates.append((sep, star_id, x, y, float(vmag),
                                   float(mag_correction), float(zenith_angle)))
    
        if not candidates:
            raise ValueError(
                f"No calibration stars found within {max_sep_px}px of target "
                f"and inside the vignetting-safe radius."
            )
    
        candidates.sort(key=lambda c: c[0])
        selected = candidates[:n_refs]
    
        # --- pass 1: centroid + Moffat fit (alpha and beta both free) ---
        centroids = []
        fits = []
        for sep, star_id, x, y, vmag, mag_correction, zenith_angle in selected:
            x_ref, y_ref = centroid_sources(self.fits_data, x, y,
                                            box_size=39, centroid_func=centroid_func)
            xc, yc = x_ref[0], y_ref[0]
            centroids.append((xc, yc, vmag, mag_correction, star_id, sep))
    
            r_field = np.hypot(xc - 4375, yc - 4375)
            pa_radial = np.arctan2(yc - 4375, xc - 4375)
    
            ab = self.fit_moffat(xc, yc)
            if ab is None:
                print(f"{star_id}  Moffat fit failed  r={r_field:.0f}")
                continue
            alpha, beta = ab
            fits.append((alpha, beta, r_field))
            print(f"{star_id}  alpha={alpha:.2f}  beta={beta:.2f}")
    
        if len(fits) < 3:
            raise ValueError(f"Only {len(fits)} usable Moffat fits for this streak.")
    
        self.moffat_fits = np.array(fits)                    # (alpha, beta, r_field)
    
        r_all = self._r_ee(self.moffat_fits[:, 0], self.moffat_fits[:, 1], EE)
        _, r_star, r_sd = sigma_clipped_stats(r_all)
        self.r_star_err = r_sd / np.sqrt(len(r_all))
        self.streak_halfwidth = self.ee_halfwidth_streak(EE=EE)
    
        a_med = np.median(self.moffat_fits[:, 0])
        r_in  = 5.0 * a_med                                  # tied to alpha, not to r_star
        r_out = 8.0 * a_med
        print(f"r_star={r_star:.2f} +/- {self.r_star_err:.2f} px  "
              f"streak halfwidth={self.streak_halfwidth:.2f} px  "
              f"annulus {r_in:.1f}-{r_out:.1f} px")
    
        # --- pass 2: photometry + zeropoints ---
        zero_points = []
        star_centres = []
        scan_stars = []
        for xc, yc, vmag, mag_correction, star_id, sep in centroids:
            flux = self.photometry_analysis(xc, yc, r_star, r_in, r_out)
            vignetted_flux, lb, ub = self.vignetting_response(flux, xc, yc)
            if not np.isfinite(vignetted_flux) or vignetted_flux <= 0:
                print(f"skipping {star_id}: flux={vignetted_flux}")
                continue
            star_centres.append((xc, yc))
            scan_stars.append((xc, yc, vmag, mag_correction))
            zero_point = vmag + 2.5 * log10(vignetted_flux)
            zero_point_corrected = zero_point + mag_correction
            print(f"zp_corrected={zero_point_corrected:.4f}  zp={zero_point:.4f}  {star_id}  "
                  f"vmag={vmag}  sep_px={sep:.1f}  flux={vignetted_flux:.1f}")
            zero_points.append(zero_point_corrected)
    
        if len(zero_points) < 3:
            print(f"Warning: only {len(zero_points)} reference stars used — "
                  f"sigma clipping may not be meaningful.")
    
        r_best, sd_best, _, _ = self.ee_radius_scan(scan_stars, r_in=r_in, r_out=r_out)
        print(f"empirical optimum {r_best:.1f} px (zp scatter {sd_best:.4f} mag) "
              f"vs model r_star={r_star:.1f}")
    
        mean, median, std = sigma_clipped_stats(np.array(zero_points), sigma=3.0, maxiters=5)
        self.plot_stars(star_centres, r_star, r_in, r_out)
        return mean, median, std, star_centres        
    
    def get_centerline(self, x, y, n_bins = 100):
        """
        Solution provided from Claude
        """
        pts = np.column_stack([x, y])
        mean = pts.mean(axis=0)
        _, _, vt = np.linalg.svd(pts - mean)
        principal = vt[0]                      # dominant direction of the trail
        proj = (pts - mean) @ principal        # 1-D coordinate along the trail
    
        order = np.argsort(proj)
        proj_sorted = proj[order]
    
        bins = np.linspace(proj_sorted.min(), proj_sorted.max(), n_bins + 1)
        centers_x, centers_y = [], []
        for i in range(n_bins):
            sel = (proj >= bins[i]) & (proj < bins[i+1])
            if sel.sum() > 0:
                centers_x.append(x[sel].mean())
                centers_y.append(y[sel].mean())
    
        return np.array(centers_x), np.array(centers_y)

        
    def gauss(self, x, amplitude, mu, sigma, ground):
        return amplitude * np.exp(-(x - mu)**2 / (2 * sigma**2)) + ground

        
    def gauss_fitting(self, counts, coordinates, plot=False):
        counts = np.asarray(counts)
        coordinates = np.asarray(coordinates)
    
        # sensible initial guesses
        ground0 = np.median(counts)
        amplitude0 = np.max(counts) - ground0
        mu0 = coordinates[np.argmax(counts)]
        sigma0 = (coordinates.max() - coordinates.min()) / 6  # rough starting width
    
        p0 = [amplitude0, mu0, sigma0, ground0]
    
        popt, pcov = curve_fit(self.gauss, coordinates, counts, p0=p0, maxfev=5000)
    
        # fit quality
        residuals = counts - self.gauss(coordinates, *popt)
        chi2 = np.sum(residuals**2)

        return popt, pcov, residuals, chi2

    
    def count_1d(self, section_img):
        section_img = np.asarray(section_img)
        counts = np.sum(section_img, axis=1)          # sum across trail width -> profile along cut
        coordinates = np.arange(len(counts))
        return counts, coordinates

        
    def sample_perpendicular(self, image, x0, y0, tangent, width=25, n=101):
        """Sample image along the line perpendicular to `tangent` at (x0,y0)."""
        normal = np.array([-tangent[1], tangent[0]])
        normal /= np.linalg.norm(normal)
        offsets = np.linspace(-width, width, n)
        xs = x0 + offsets * normal[0]
        ys = y0 + offsets * normal[1]
        profile = map_coordinates(image, [ys, xs], order=1)  # bilinear interp
        return offsets, profile        

    
    def build_epsf_model(self, n_refs=400, max_sep_px=None, oversampling=4,
                          cutout_size=25, maxiters=10):
        """
        Build an empirical PSF model from reference stars using photutils,
        following the Anderson & King (2000) effective-PSF formalism.
    
        n_refs: how many candidate stars to attempt to use (e.g. all 400,
                or a smaller local subset for a per-region/per-streak model)
        max_sep_px: optional cap on distance from field center (None = no cap;
                    pass a value if you're building a *local* ePSF instead
                    of the full-field one)
        cutout_size: stamp size in pixels (must be odd; keep it well clear
                     of your source_radius/annulus so the wings are captured)
        """
        centroid_func = centroid_2dg
        candidates = []
    
        with open("stellar_calibration.txt") as f:
            for line in f:
                star_id, ra, dec, vmag, bmag, sp = line.strip().split(",")
                ra_arr = np.atleast_1d(float(ra))
                dec_arr = np.atleast_1d(float(dec))
                x_pred, y_pred = raDecToXYPP(ra_arr, dec_arr, self.exposure_start_jd, self.pp)
                x, y = x_pred[0], y_pred[0]
    
                # same vignetting-safe radius cut you already use
                radius = sqrt((x - 4375)**2 + (y - 4375)**2)
                if radius >= 3500:
                    continue
    
                if max_sep_px is not None:
                    # e.g. distance from a chosen field/fiducial point rather
                    # than frame center, if this is a per-region call
                    if radius > max_sep_px:
                        continue
    
                candidates.append((star_id, x, y))
    
        if len(candidates) < 10:
            raise ValueError(
                f"Only {len(candidates)} candidate stars available — "
                f"too few for a stable EPSFBuilder fit."
            )
    
        candidates = candidates[:n_refs]
    
        # Refine centroids the same way you already do for calibration stars
        xs, ys = [], []
        for star_id, x, y in candidates:
            x_ref, y_ref = centroid_sources(
                self.fits_data, np.atleast_1d(x), np.atleast_1d(y),
                box_size=19, centroid_func=centroid_func
            )
            xs.append(x_ref[0])
            ys.append(y_ref[0])
    
        stars_tbl = Table()
        stars_tbl['x'] = xs
        stars_tbl['y'] = ys
    
        nddata = NDData(data=self.fits_data)
        star_stamps = extract_stars(nddata, stars_tbl, size=cutout_size)
    
        epsf_builder = EPSFBuilder(
            oversampling=oversampling,
            maxiters=maxiters,
            progress_bar=True,
            smoothing_kernel='quartic',   # matches A&K's smoothing kernel, eq. 8
        )
        epsf, fitted_stars = epsf_builder(star_stamps)
    
        self.epsf = epsf              # photutils.psf.EPSFModel — evaluable at (x, y)
        self.epsf_fitted_stars = fitted_stars
        return epsf, fitted_stars


        
    def streak_gaussian_analysis_curved(self, image, tck, arc_step=50, perp_width=25):
        # dense sample to compute cumulative arc length
        u_fine = np.linspace(0, 1, 2000)
        pts = np.array(splev(u_fine, tck)).T
        seglen = np.sqrt(np.sum(np.diff(pts, axis=0)**2, axis=1))
        arc = np.concatenate([[0], np.cumsum(seglen)])
        total_length = arc[-1]
    
        step_positions = np.arange(0, total_length, arc_step)
        section, popts, chi2s = [], [], []
    
        for s in step_positions:
            u_s = np.interp(s, arc, u_fine)
            x0, y0 = splev(u_s, tck)
            dx, dy = splev(u_s, tck, der=1)
            tangent = np.array([dx, dy]) / np.hypot(dx, dy)
    
            offsets, counts = self.sample_perpendicular(image, x0, y0, tangent, width=perp_width)
    
            try:
                popt, _, _, chi2 = self.gauss_fitting(counts, offsets, False)
                popts.append(popt)
                section.append(s)
                chi2s.append(chi2)
            except RuntimeError:
                pass
    
        section = np.array(section)
        amplitude = np.array([p[0] for p in popts])
        mu        = np.array([p[1] for p in popts])
        sigma     = np.array([p[2] for p in popts])
        ground    = np.array([p[3] for p in popts])
        chi2      = np.array(chi2s)
    
        return section, amplitude, ground, mu, sigma, chi2

    
    def select_mask(self, spline_coords, streak_coords):
        spline_coords = np.asarray(spline_coords)
        satellite_coords = np.asarray(streak_coords)

        if len(spline_coords) < 2:
            return np.inf

        tree_spline = cKDTree(spline_coords)
        tree_satellite = cKDTree(satellite_coords)

            # distance from each satellite point to nearest spline point
        d_satellite_to_spline, _ = tree_spline.query(satellite_coords)
        # distance from each spline point to nearest satellite point
        d_spline_to_satellite, _ = tree_satellite.query(spline_coords)

        all_d = np.concatenate([d_satellite_to_spline, d_spline_to_satellite])

        mean_dist = np.mean(all_d)
        spread = np.std(all_d)
    
        # weight can be tuned - spread penalizes tracks that don't match the streak's shape/length
        score = mean_dist + spread
        return score, mean_dist, spread





    def inverse_gnomonic(self, xi, eta, ra_centre_deg, dec_centre_deg):
        """
        Inverse gnomonic projection. Valid for xi, eta < pi/2 radians!
        """
        # Converts degress -> radians
        ra0 = np.radians(ra_centre_deg) 
        dec0 = np.radians(dec_centre_deg)
        
        rho = np.sqrt(xi**2 + eta**2)
        c = np.arctan(rho)
        sin_c = np.sin(c)
        cos_c = np.cos(c)

        with np.errstate(invalid="ignore", divide = "ignore"):
            dec = np.where(
                rho > 1e-12, # Stops divide by 0 errors!
                np.arcsin(cos_c * np.sin(dec0) + (eta * sin_c * np.cos(dec0)) / rho),
                dec0,
            )
            ra = np.where(
                rho > 1e-12,
                ra0 + np.arctan2(
                    xi * sin_c,
                    rho * np.cos(dec0) * cos_c - eta * np.sin(dec0) * sin_c,
                ),
                ra0,
            )
            return np.degrees(ra) % 360.0, np.degrees(dec) 
    def get_obs_time_tuple(self):
        dt = self.exposure_start.datetime
        return (dt.year, dt.month, dt.day, dt.hour, dt.minute, dt.second, dt.microsecond // 1000)
    def great_circle_destination(self, ra0_deg, dec0_deg, bearing_deg, ang_dist_deg):
        ra0 = np.radians(ra0_deg)
        dec0 = np.radians(dec0_deg)
        brng = np.radians(bearing_deg)
        d = np.radians(ang_dist_deg)
    
        dec = np.arcsin(np.sin(dec0) * np.cos(d) + np.cos(dec0) * np.sin(d) * np.cos(brng))
        ra = ra0 + np.arctan2(
            np.sin(brng) * np.sin(d) * np.cos(dec0),
            np.cos(d) - np.sin(dec0) * np.sin(dec),
        )
        return np.degrees(ra) % 360.0, np.degrees(dec)
    def cutoff_boundary_pixels(self, pp, ra_centre_deg, dec_centre_deg, obs_time_tuple, cutoff_deg, n_points=720):
        bearings = np.linspace(0, 360, n_points, endpoint=False)
        ra_line, dec_line = self.great_circle_destination(ra_centre_deg, dec_centre_deg, bearings, cutoff_deg)
        jd = date2JD(*obs_time_tuple)
        x_pix, y_pix = raDecToXYPP(ra_line, dec_line, jd, pp)
        return x_pix, y_pix  
    def crop_fits_by_angle(self, fits_out_path, cutoff_deg=75.0,
                            preview_png="crop_preview.png"):
        """
        Crops the raw fits exposure to eliminate buildings and ensure that gnomonic map doesnt explode at 90degrees.
        """
        
        height, width = self.fits_data.shape[-2], self.fits_data.shape[-1]
        ra_centre_deg, dec_centre_deg = self.ra, self.dec
        obs_time_tuple = self.get_obs_time_tuple()
    
        x_bound, y_bound = self.cutoff_boundary_pixels(
            self.pp, ra_centre_deg, dec_centre_deg, obs_time_tuple, cutoff_deg
        )
    
        # --- preview before committing to the crop ---
        vmin, vmax = np.nanpercentile(self.fits_data, [1, 99.5])
        
        # --- build mask from the boundary polygon ---
        mask = np.zeros((height, width), dtype=np.uint8)
        poly = np.stack([x_bound, y_bound], axis=1).round().astype(np.int32).reshape(-1, 1, 2)
        cv2.fillPoly(mask, [poly], 1)
    
        # --- zero out everything outside the mask ---
        masked_data = self.fits_data.copy()
        masked_data[mask == 0] = 0
    
        # --- crop to the mask's bounding box ---
        ys, xs = np.where(mask == 1)
        y0, y1 = int(ys.min()), int(ys.max())
        x0, x1 = int(xs.min()), int(xs.max())
        cropped = masked_data[y0:y1 + 1, x0:x1 + 1]
    
        print(f"Original size: {width} x {height}")
        print(f"Cropped size:  {x1 - x0 + 1} x {y1 - y0 + 1}")
        print(f"Crop offset (add back before using the platepar again): X0={x0}, Y0={y0}")
        header = self.header.copy()
        # --- record the crop offset in the header for later use ---
        header["HISTORY"] = f"Cropped to {cutoff_deg} deg from boresight (RA={ra_centre_deg:.4f}, Dec={dec_centre_deg:.4f})"
        header["CROP_X0"] = (x0, "X offset of this crop within the original frame")
        header["CROP_Y0"] = (y0, "Y offset of this crop within the original frame")
        header["CROP_ANG"] = (cutoff_deg, "Angular radius (deg) used for this crop")

        self.x_offset = x0
        self.y_offset = y0
        self.cropped_data = cropped # will adjust ofc!
        self.cutoff_deg = cutoff_deg
        
        fits.writeto(fits_out_path, cropped, header, overwrite=True)
        print(f"Wrote cropped FITS: {fits_out_path}")
    
        return fits_out_path
    def run_crop_procedure(self):
        self.crop_fits_by_angle(
            fits_out_path=f"{self.file_name}_cropped67deg.fits",
            cutoff_deg=67.0,  # pick anywhere in your 65-70 deg range
        )
    def undistorted_xy_to_original_xy(self, px, py, out_width, out_height, half_tan,
                                       ra_centre_deg, dec_centre_deg, pp, obs_time_tuple,
                                       crop_x0=0, crop_y0=0):
        """
    
        """
        px = np.atleast_1d(np.asarray(px, dtype=np.float64))
        py = np.atleast_1d(np.asarray(py, dtype=np.float64))
    
        xi = (px / (out_width - 1) - 0.5) * 2 * half_tan
        eta = (py / (out_height - 1) - 0.5) * 2 * half_tan
    
        ra_deg, dec_deg = self.inverse_gnomonic(xi, eta, ra_centre_deg, dec_centre_deg)
    
        jd = date2JD(*obs_time_tuple)
        x_orig, y_orig = raDecToXYPP(ra_deg, dec_deg, jd, pp)
    
        # x_orig, y_orig are in the ORIGINAL (uncropped) frame's coordinates.
        # Convert to the CROPPED frame's coordinates (what we actually remap from).
        return x_orig - crop_x0, y_orig - crop_y0
    def build_dense_undistortion_map(self, out_width, out_height, half_tan, ra_centre_deg, dec_centre_deg,
                                      pp, obs_time_tuple, crop_x0, crop_y0):
        yy, xx = np.mgrid[0:out_height, 0:out_width]
        x_crop, y_crop = self.undistorted_xy_to_original_xy(
            xx.ravel(), yy.ravel(), out_width, out_height, half_tan,
            ra_centre_deg, dec_centre_deg, pp, obs_time_tuple, crop_x0, crop_y0,
        )
        map_x = x_crop.reshape(out_height, out_width).astype(np.float32)
        map_y = y_crop.reshape(out_height, out_width).astype(np.float32)
        return map_x, map_y
    def undistort_cropped_fits(self, cropped_fits_path, fits_out_path,
                                out_width=None, out_height=None):
        
        with fits.open(cropped_fits_path, memmap=False) as hdul:
            header = hdul[0].header.copy()
    
        ra_centre_deg, dec_centre_deg = self.ra, self.dec
        obs_time_tuple = self.get_obs_time_tuple()
    
        half_tan = np.tan(np.radians(self.cutoff_deg))
    
        if out_width is None:
            out_width = self.cropped_data.shape[-1]
        if out_height is None:
            out_height = self.cropped_data.shape[-2]
    
        print(f"Building dense analytic undistortion map: {out_width} x {out_height} pixels...")
        map_x, map_y = self.build_dense_undistortion_map(
            out_width, out_height, half_tan, ra_centre_deg, dec_centre_deg,
            self.pp, obs_time_tuple, self.x_offset, self.y_offset,
        )
    
        corrected = cv2.remap(self.cropped_data.astype(np.float32), map_x, map_y, interpolation=cv2.INTER_CUBIC)
        self.corrected_data = corrected.copy()
        # Save everything needed to invert this later (streak -> distorted frame)
        header["HISTORY"] = "Dense analytic undistortion (tangent-plane / gnomonic)"
        header["UND_RA0"] = (ra_centre_deg, "Tangent point RA (deg) used for undistortion")
        header["UND_DEC0"] = (dec_centre_deg, "Tangent point Dec (deg) used for undistortion")
        header["UND_HTAN"] = (half_tan, "Half-width of tangent-plane extent (tan(cutoff_deg))")
        header["UND_W"] = (out_width, "Undistorted image width used to build this map")
        header["UND_H"] = (out_height, "Undistorted image height used to build this map")
        # CROP_X0/CROP_Y0 already present from the crop step -- keep them; they're
        # still needed to map all the way back to the ORIGINAL uncropped frame.
    
        fits.writeto(fits_out_path, corrected, header, overwrite=True)
        print(f"Wrote undistorted FITS: {fits_out_path}")
        return fits_out_path
    def undistort_fits(self):
        self.undistort_cropped_fits(
            cropped_fits_path=f"{self.file_name}_cropped67deg.fits",
            fits_out_path=f"{self.file_name}_undistorted.fits",
        )  
    def remove_background(self, img, sigma=3, maxiters=10, kernel_size=(70, 70), filter_size=(3, 3)):
        sigma_clip = SigmaClip(sigma, maxiters=maxiters)
        bkg = Background2D(
            img, kernel_size, filter_size=filter_size,
            sigma_clip=sigma_clip, bkg_estimator=MedianBackground(),
        )
        return img - bkg.background
    def preprocess(self, img, brightness_cuts=(1, 1), thresholding_cut=0.5,
                   flux_prop_thresholds=(0.1, 0.2, 0.3, 1.0), blur_kernel_sizes=(3, 5, 9, 11),
                   canny_thresholds=(0, 200)):
        img = self.remove_background(img.astype(np.float64))
    
        up_limit = img.mean() + brightness_cuts[1] * img.std()
        low_limit = img.mean() - brightness_cuts[0] * img.std()
        img = np.clip(img, None, up_limit)
        img[img <= low_limit] = 0
    
        norm = (img - img.mean()) / (img.std() + 1e-9)
        norm -= norm.min()
        norm = 255 * (norm - norm.min()) / (norm.max() - norm.min() + 1e-9)
    
        limit = norm.mean() + thresholding_cut * norm.std()
        _, thresholded = cv2.threshold(norm.astype(np.float32), limit, 255, cv2.THRESH_BINARY)
        thresholded = cv2.convertScaleAbs(thresholded)
    
        prop_bright = np.mean(thresholded > np.mean(thresholded))
        kernel = blur_kernel_sizes[-1]
        for thresh_frac, k in zip(flux_prop_thresholds, blur_kernel_sizes):
            if prop_bright < thresh_frac:
                kernel = k
                break
    
        blurred = cv2.medianBlur(thresholded, kernel)
        edges = cv2.Canny(blurred, *canny_thresholds)
    
        return thresholded, blurred, edges
    def detect_segments(self, edges, rho=1, theta_deg=0.5, hough_threshold=50,
                         min_line_length=200, max_line_gap=10):
        theta = np.radians(theta_deg)
        segments = cv2.HoughLinesP(
            edges, rho, theta, hough_threshold,
            minLineLength=min_line_length, maxLineGap=max_line_gap,
        )
        if segments is None:
            return np.empty((0, 4))
        return segments.reshape(-1, 4).astype(np.float64)
    def detect_blob_lines(self, thresholded, min_area=100, min_elongation=5):
        contours, _ = cv2.findContours(thresholded, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        segments = []
        for c in contours:
            if cv2.contourArea(c) < min_area:
                continue
            (cx, cy), (w, h), angle = cv2.minAreaRect(c)
            length, width = max(w, h), max(min(w, h), 1e-3)
            if length / width < min_elongation:
                continue  # too round/blob-like to be a streak (e.g. a star)
    
            vx, vy, x0, y0 = cv2.fitLine(c.reshape(-1, 2).astype(np.float32),
                                          cv2.DIST_L2, 0, 0.01, 0.01).flatten()
            half_len = length / 2
            segments.append((x0 - vx * half_len, y0 - vy * half_len,
                              x0 + vx * half_len, y0 + vy * half_len))
    
        return np.array(segments) if segments else np.empty((0, 4))
    def segment_to_hough_params(self, x1, y1, x2, y2):
        line_angle = np.arctan2(y2 - y1, x2 - x1)
        theta_n = (line_angle - np.pi / 2) % np.pi
        dist = x1 * np.cos(theta_n) + y1 * np.sin(theta_n)
        return dist, theta_n
    def merge_duplicate_segments(self, segments, dist_tol=15, angle_tol_deg=5):
        if len(segments) == 0:
            return segments
    
        params = np.array([self.segment_to_hough_params(*s) for s in segments])
        used = np.zeros(len(segments), dtype=bool)
        merged = []
    
        for i in range(len(segments)):
            if used[i]:
                continue
            group_idx = [i]
            used[i] = True
            for j in range(i + 1, len(segments)):
                if used[j]:
                    continue
                d_diff = abs(params[i, 0] - params[j, 0])
                a_diff = abs(params[i, 1] - params[j, 1])
                a_diff = min(a_diff, np.pi - a_diff)
                if d_diff < dist_tol and np.degrees(a_diff) < angle_tol_deg:
                    group_idx.append(j)
                    used[j] = True
    
            group = segments[group_idx]
            lengths = np.hypot(group[:, 2] - group[:, 0], group[:, 3] - group[:, 1])
            merged.append(group[np.argmax(lengths)])
    
        return np.array(merged)
    def detect_and_convert(self, undistorted_fits_path,
                            preprocess_kwargs=None, hough_kwargs=None,
                            merge_kwargs=None, n_samples_per_segment=200):
    
        with fits.open(undistorted_fits_path, memmap=False) as hdul:
            undistorted_data = hdul[0].data.copy()
            header = hdul[0].header.copy()
    
        ra_centre_deg = header["UND_RA0"]
        dec_centre_deg = header["UND_DEC0"]
        half_tan = header["UND_HTAN"]
        out_width = header["UND_W"]
        out_height = header["UND_H"]
        crop_x0 = header.get("CROP_X0", 0)
        crop_y0 = header.get("CROP_Y0", 0)
        obs_time_tuple = self.get_obs_time_tuple()
    
        preprocess_kwargs = preprocess_kwargs or {}
        hough_kwargs = hough_kwargs or {}
        merge_kwargs = merge_kwargs or {}
    
        thresholded, blurred, edges = self.preprocess(undistorted_data, **preprocess_kwargs)
    
        raw_segments_hough = self.detect_segments(edges, **hough_kwargs)
        raw_segments_blob = self.detect_blob_lines(thresholded)
        raw_segments = np.vstack([raw_segments_hough, raw_segments_blob])
        print(f"HoughLinesP found {len(raw_segments_hough)}, blob detector found "
              f"{len(raw_segments_blob)} raw segment(s).")
    
        segments = self.merge_duplicate_segments(raw_segments, **merge_kwargs)
        print(f"After merging near-duplicates: {len(segments)} streak(s).")
    
        results = []
        for i, (x1, y1, x2, y2) in enumerate(segments):
            x_seg = np.linspace(x1, x2, n_samples_per_segment)
            y_seg = np.linspace(y1, y2, n_samples_per_segment)
    
            x_crop_dist, y_crop_dist = self.undistorted_xy_to_original_xy(
                x_seg, y_seg, out_width, out_height, half_tan,
                ra_centre_deg, dec_centre_deg, self.pp, obs_time_tuple,
                crop_x0=crop_x0, crop_y0=crop_y0,
            )
            x_original_dist = x_crop_dist + crop_x0
            y_original_dist = y_crop_dist + crop_y0
            discrim_radius = 3000
            length_px = np.hypot(x2 - x1, y2 - y1)

            distance = np.sqrt((x2 - 3246)**2+(y2 - 3246)**2) # HARDCODED PLEASE CHANGE TO IMPROVE FUNCTIONALITY OSCAR!!!!
            
            if length_px > 200:
                if distance < 3000:
                    print(f"  Streak {i}: undistorted endpoints ({x1:.0f},{y1:.0f}) -> "
                          f"({x2:.0f},{y2:.0f}), length ~{length_px:.1f} px")
                        
                    results.append({
                        "undistorted_endpoints": (x1, y1, x2, y2),
                        "undistorted_xy": (x_seg, y_seg),
                        "cropped_frame_xy": (x_crop_dist, y_crop_dist),
                        "original_frame_xy": (x_original_dist, y_original_dist),
                    })
        return results, {"thresholded": thresholded, "blurred": blurred, "edges": edges}
    def run_hough_transform(self):
        UNDISTORTED_FITS = f"{self.file_name}_undistorted.fits"
        CROPPED_FITS = f"{self.file_name}_cropped67deg.fits"
        
        results, intermediates = self.detect_and_convert(
            undistorted_fits_path=UNDISTORTED_FITS,
            hough_kwargs=dict(rho=1, theta_deg=0.5, hough_threshold=50,
                               min_line_length=200, max_line_gap=10),
            merge_kwargs=dict(dist_tol=40, angle_tol_deg=5),
        )
        print(f"\n{len(results)} streak(s) with recovered segments and distorted-frame paths.")
    
        with fits.open(UNDISTORTED_FITS, memmap=False) as hdul:
            undistorted_data = hdul[0].data.copy()
    
        with fits.open(CROPPED_FITS, memmap=False) as hdul:
            cropped_original_data = hdul[0].data.copy()

       
        #self.plot_streaks(undistorted_data, results)
        if self.simulated:
            self.analyse_simulated_streak(
            results,
            self.ra_truth_start, self.dec_truth_start,
            self.ra_truth_end, self.dec_truth_end,
            original_data=self.fits_data
            )
        else:
            self.plot_streaks(undistorted_data, results, original_data=self.fits_data)


         
    # The following code needs some serious work!
    def plot_streaks(self, undistorted_data, results, original_data=None):
        if original_data is not None:
            i = 0
            for r in results:
                refined = self.refine_streak_centerline(
                *r["original_frame_xy"],   # x_seg, y_seg (after your crop_x0 fix)
                perp_halfwidth=10,
                    
                )
                
                x_seg, y_seg = refined['x_refined'], refined['y_refined']

                points = np.column_stack([x_seg, y_seg])
    
                inliers, model = self.ransac_streak_spline_fitting(points = points, order=2)


                cx, cy = self.get_centerline(x_seg, y_seg, n_bins=40)
                tck, u = splprep([cx, cy], s=len(cx) * 10, k=2)
                
                u_check = np.linspace(0, 1, 500)
                
                spline_xs, spline_ys = splev(u_check, tck)

                
                
                x_mid, y_mid = splev(0.5, tck)
                
                _, ra_c, dec_c, _ = xyToRaDecPP(
                [jd2Date(self.exposure_start_jd)], [x_mid], [y_mid], [1], self.pp
                )

                spline_xy = np.column_stack([spline_xs, spline_ys])
                print(ra_c, dec_c)

                try:
                    satellites = self.streak_passes(float(ra_c[0]), float(dec_c[0]), radius = 5)
                except Exception as e:
                    print("Streak passes error, there appears to be no data for that query.")
                    satellites = {}

                    
                mean, median, std, star_centres = self.stellar_calibrations(location = (ra_c, dec_c))
                
                candidates = []
                
                lengths, centres, theta = self.split_streak_into_sections(spline_xs, 
                                                                          spline_ys)
                
                for sat_name, positions in satellites.items():
                    sat_xs, sat_ys, times = [], [], []
                
                    for pos in positions:
                        x_arr, y_arr, ra, dec, jd, range_km, norad_id = pos
                        
                        if range_km < 2000:
                            
                            sat_xs.append(x_arr[0])   # unwrap the 1-element array, shift to local coords
                            sat_ys.append(y_arr[0])
                            times.append(jd)
                
                    #if sat_xs:  # only plot if something passed the range filter
                      #  ax2.plot(sat_xs, sat_ys, label=f"{sat_name} - {i}")
                       # ax2.scatter(sat_xs, sat_ys, s=6)
                        
                    if len(sat_xs) < 2:
                        continue
    
                    sat_xy = np.column_stack([sat_xs, sat_ys])
                    
                    score, mean_dist, spread = self.select_mask(spline_xy, sat_xy)
                    
                    candidates.append((score, sat_name, sat_xy, mean_dist, spread, norad_id))
                    candidates.sort(key=lambda c: c[0])
            
                for score, sat_name, sat_xy, mean_dist, spread, norad_id in candidates:
                    print(f"{sat_name}: score={score:.2f}  mean_dist={mean_dist:.2f}  spread={spread:.2f}")
                    
                fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize = (40, 40))
                
                if candidates:
                    best_score, best_name, best_xy, best_mean_dist, best_mean_spread, best_norad_id = candidates[0]
                    ax2.plot(best_xy[:,0], best_xy[:,1], label=f"{best_name} (best match)", linewidth = 3)
                    ax2.scatter(best_xy[:,0], best_xy[:,1], s=6)
                    ax2.set_title(f"Most likely satellite match: {best_name}")
                else:
                    best_norad_id = None
                    ax2.set_title("No satellite match found")
                
                ax1.plot(x_seg, y_seg, color="lime", linewidth=1.5)
                vmin, vmax = np.percentile(self.fits_data, [1, 99.5])
                ax1.imshow(self.fits_data, origin='upper', cmap='gray', vmin=vmin, vmax=vmax)
                ax1.set_xlim(np.min(x_seg) - 100, np.max(x_seg) + 100)
                ax1.set_ylim(np.min(y_seg) - 100, np.max(y_seg) + 100)
                
                ax2.imshow(self.fits_data, origin='upper', cmap='gray', vmin=vmin, vmax=vmax)
                ax2.plot(spline_xs, spline_ys, color = "blue", linewidth = 4, label = "Real Streak")
                ax2.set_xlim(np.min(x_seg) - 100, np.max(x_seg) + 100)
                ax2.set_ylim(np.min(y_seg) - 100, np.max(y_seg) + 100)
                ax2.grid()
                
                ax3.plot(refined["x_refined"], refined["y_refined"], color="lime", label="Refined", linewidth=1.5)

                x_range = np.linspace(points[:, 0].min(), points[:, 0].max(), 400)
                y_fit = model.predict(x_range)   

                
                ax3.plot(spline_xs, spline_ys, color = "blue", linewidth = 1)
                ax3.imshow(self.fits_data, origin='upper', cmap='gray', vmin=vmin, vmax=vmax)
                ax3.set_xlim(np.min(x_seg) - 100, np.max(x_seg) + 100)
                ax3.set_ylim(np.min(y_seg) - 100, np.max(y_seg) + 100)
                i+=1
                plt.savefig(f"streak_{i}.png")
                plt.show()
                
                if best_norad_id is not None:
                    ratio = self.computing_ratio_of_lengths(start_point = (spline_xs[0], spline_ys[0]),
                                                    end_point = (spline_xs[-1], spline_ys[-1]), satellite_nid = best_norad_id)
                else:
                    ratio = 1.0
                    print("There was no satellite to compare to!")
                
                mag_arcsec, mag_total, flux_net_list, mag_120_normalised = self.photometry_of_streak(lengths = lengths, 
                                                                                 centres = centres, theta = theta, 
                                                                                 stellar_calib_baseline = mean, 
                                                                                 ratio = ratio, zp_error = std)
                
                #psf_mag_arcsec, psf_mag_total, psf_flux_net_list, flux_err_list, final_mask = self.psf_photometry_on_streaks(lengths = lengths, 
                 #                                                                   centres = centres, theta = theta, stellar_calib_baseline = psf_mean)
                
                print(f"Aperture mag/arcmin: {mag_arcsec}")
                #print(f"PSF mag/arcmin: {psf_mag_arcsec}")
                print(f"Aperture raw mag: {mag_total}")
                #print(f"PSF raw mag: {psf_mag_total}")  
                print(f"Aperture 120s normalised mag: {mag_120_normalised}")

    def _epsf_perp_profile_fit(self, x0, y0, px, py, perp_offsets, profile, epsf=None):
        """
        Fit a candidate perpendicular offset (mu) by matching the ePSF's
        cross-track profile to the observed data, rather than assuming a
        Gaussian shape. Amplitude + background are linear given mu; mu itself
        is found by a small bounded scalar search (analogous to the A&K
        eq. 12 Newton step, but restricted to the perpendicular direction).
        """
        epsf = epsf or self.epsf
    
        def residual_sumsq(mu):
            xc = x0 + mu * px
            yc = y0 + mu * py
            xs = x0 + perp_offsets * px
            ys = y0 + perp_offsets * py
            model = epsf.evaluate(xs, ys, flux=1.0, x_0=xc, y_0=yc)
    
            # linear least-squares for amplitude + background given this mu
            A = np.vstack([model, np.ones_like(model)]).T
            coeffs, *_ = np.linalg.lstsq(A, profile, rcond=None)
            amp, bkg = coeffs
            resid = profile - (amp * model + bkg)
            return np.sum(resid**2), amp, bkg, model
    
        # bounded scalar search over the same window used for sampling
        from scipy.optimize import minimize_scalar
        res = minimize_scalar(
            lambda mu: residual_sumsq(mu)[0],
            bounds=(perp_offsets.min(), perp_offsets.max()),
            method='bounded'
        )
        mu_fit = res.x
        _, amp_fit, _, model_at_fit = residual_sumsq(mu_fit)
        return mu_fit, amp_fit, model_at_fit    
        
    def refine_streak_centerline(self, x_seg, y_seg, image=None,
                              perp_halfwidth=4, along_step=5.0,
                              fit_method='gaussian',  # 'gaussian' | 'epsf' | 'centroid'
                              epsf=None, mad_thresh=3.0,
                              min_snr=2.0):
        """
        Refine a coarse Hough-derived streak polyline (x_seg, y_seg) into a
        sub-pixel-accurate centerline, by sampling perpendicular cross-sections
        of the RAW (undistorted-projection-free) image data along the prior
        path and centroiding each one.
        
        Parameters
        ----------
        x_seg, y_seg : array-like
            Coarse path in ORIGINAL (distorted) frame pixel coords -- e.g.
            results[i]["original_frame_xy"] after the crop_x0/crop_y0 fix.
        image : 2D array, optional
            Background-subtracted image to sample. Defaults to
            self.remove_background(self.fits_data).
        perp_halfwidth : float
            Half-width (px) of the perpendicular search/sampling window.
            Keep this tight (a few px) -- this is your main defense against
            the fit wandering onto a nearby star.
        along_step : float
            Spacing (px) between along-track sampling points. Finer than 1 px
            is fine since you're just resampling a smooth prior line.
        use_gaussian : bool
            If True, fit a 1D Gaussian to each cross-section for sub-pixel
            centroid + amplitude + width. Falls back to flux-weighted centroid
            if the fit fails to converge.
        mad_thresh : float
            Outlier rejection threshold (multiples of MAD) on perpendicular
            offset from the smooth prior -- flags likely star contamination.
        min_snr : float
            Minimum peak amplitude / local noise ratio to trust a cross-section
            at all; below this it's treated as background/gap and interpolated.
    
        Returns
        -------
        dict with:
            'x_refined', 'y_refined'   : sub-pixel centerline (bad points
                                          interpolated from good neighbors)
            'amplitude'                : per-step fitted/measured peak flux
            'width'                    : per-step fitted Gaussian sigma (nan
                                          if use_gaussian=False or fit failed)
            'flagged'                  : boolean mask, True where a point was
                                          rejected as an outlier / low-SNR and
                                          had to be interpolated
            'perp_offset_raw'          : raw perpendicular offset (px) from the
                                          prior line at each step, pre-rejection
        """
        if image is None:
            image = self.remove_background(self.fits_data.astype(np.float64))
    
        x_seg = np.asarray(x_seg, dtype=np.float64)
        y_seg = np.asarray(y_seg, dtype=np.float64)
    
        # --- 1. Resample the coarse prior to even along-track spacing ---
        dx = np.diff(x_seg)
        dy = np.diff(y_seg)
        seg_lengths = np.hypot(dx, dy)
        cum_len = np.concatenate([[0], np.cumsum(seg_lengths)])
        total_len = cum_len[-1]
        if total_len < along_step:
            raise ValueError("Streak segment too short to refine.")
    
        n_steps = max(int(total_len / along_step), 2)
        s_new = np.linspace(0, total_len, n_steps)
        x_prior = np.interp(s_new, cum_len, x_seg)
        y_prior = np.interp(s_new, cum_len, y_seg)
    
        # local tangent direction at each point (finite difference, smoothed)
        tx = np.gradient(x_prior)
        ty = np.gradient(y_prior)
        tnorm = np.hypot(tx, ty)
        tnorm[tnorm == 0] = 1.0
        tx, ty = tx / tnorm, ty / tnorm
        # perpendicular unit vector
        px_, py_ = -ty, tx
    
        # --- 2. Sample perpendicular cross-sections and centroid each one ---
        perp_offsets = np.linspace(-perp_halfwidth, perp_halfwidth, int(4 * perp_halfwidth) + 1)
    
        x_refined = np.full(n_steps, np.nan)
        y_refined = np.full(n_steps, np.nan)
        amplitude = np.full(n_steps, np.nan)
        width = np.full(n_steps, np.nan)
        perp_offset_raw = np.full(n_steps, np.nan)
    
        def gauss1d(u, amp, mu, sigma, offset):
            return offset + amp * np.exp(-0.5 * ((u - mu) / sigma) ** 2)
    
        for i in range(n_steps):
            xs = x_prior[i] + perp_offsets * px_[i]
            ys = y_prior[i] + perp_offsets * py_[i]
            profile = map_coordinates(image, [ys, xs], order=1, mode='nearest')

            local_noise = np.std(profile) + 1e-6
            peak = profile.max()

            if peak / local_noise < min_snr:
                continue

            if fit_method == 'gaussian':
                try:
                    p0 = [peak - profile.min(), perp_offsets[np.argmax(profile)],
                          max(perp_halfwidth / 3, 1.0), np.median(profile)]
                    popt, _ = curve_fit(gauss1d, perp_offsets, profile, p0=p0, maxfev=1000)
                    amp_fit, mu_fit, sigma_fit, _ = popt
                    if not (-perp_halfwidth <= mu_fit <= perp_halfwidth) or sigma_fit <= 0:
                        raise RuntimeError("Fit landed outside window / degenerate.")
                except Exception:
                    w = profile - profile.min()
                    mu_fit = np.sum(perp_offsets * w) / (np.sum(w) + 1e-9)
                    amp_fit = peak
                    sigma_fit = np.nan
                    
            else:  # 'centroid'
                w = profile - profile.min()
                mu_fit = np.sum(perp_offsets * w) / (np.sum(w) + 1e-9)
                amp_fit = peak
                sigma_fit = np.nan

            x_refined[i] = x_prior[i] + mu_fit * px_[i]
            y_refined[i] = y_prior[i] + mu_fit * py_[i]
            amplitude[i] = amp_fit
            width[i] = sigma_fit
            perp_offset_raw[i] = mu_fit

        mask = self.hampel_with_persistence(perp_offset_raw, window=50, n_sigmas=3, min_run=20)
        #y = np.asarray(perp_offset_raw, dtype = float)
        #x_axis = np.arange(0, len(perp_offset_raw), 1)
        #plt.plot(x_axis, y, color='tab:blue', label='Flux', markersize=3)
        #plt.scatter(x_axis[~mask], y[~mask], 
        #            marker='x', color='red', s=80, zorder=5, label='Rejected outliers')
        #plt.grid(True)
        #plt.legend()
        #plt.show()
        #plt.show()
        # --- 3. Outlier rejection: reject points whose perpendicular offset ---
        #     deviates too far from the smooth local trend (catches stars/glitches)
        flagged = np.isnan(x_refined)  # already-failed (low SNR) points
    
        valid = ~np.isnan(perp_offset_raw)
        if valid.sum() >= 5:
            # smooth trend via median filter over a small window
            from scipy.ndimage import median_filter
            trend = np.full(n_steps, np.nan)
            trend[valid] = median_filter(perp_offset_raw[valid], size=min(9, valid.sum()), mode='nearest')
            resid = np.abs(perp_offset_raw - trend)
            mad = np.nanmedian(np.abs(resid[valid] - np.nanmedian(resid[valid]))) + 1e-9
            outliers = valid & (resid > mad_thresh * 1.4826 * mad)
            flagged |= outliers
            x_refined[outliers] = np.nan
            y_refined[outliers] = np.nan
    
        # --- 4. Interpolate over flagged/NaN points so the centerline stays continuous ---
        good = ~np.isnan(x_refined)
        if good.sum() < 2:
            raise RuntimeError("Refinement failed -- too few good cross-sections "
                                "(streak too faint, or entirely contaminated).")
    
        idx = np.arange(n_steps)
        x_refined = np.interp(idx, idx[good], x_refined[good])
        y_refined = np.interp(idx, idx[good], y_refined[good])
    
        return {
            "x_refined": x_refined,
            "y_refined": y_refined,
            "amplitude": amplitude,
            "width": width,
            "flagged": flagged,
            "perp_offset_raw": perp_offset_raw,
            "s": s_new,  # along-track distance, useful for plotting flux vs position
        }

        
    def second_refinedment_stage(self, spline_x, spline_y, fit_radius = 2, residual_threshold = 1.5):
        pass

    
    def using_the_streak(self, results):
        """
        This function allows me to access the streak
        """
        width = results['perp_offset_raw']
        med = np.median(width)
        std = np.std(width)
        outlier_idx = np.where(np.abs(width - med) > 2 * std)[0]
        print(outlier_idx)        

        
    def split_streak_into_sections(self, x, y, step = 20):
        """
        Given ordered x, y coordinates along a curve, return points spaced
        approximately `step` pixels apart along the path (arc length).
        """
        x = np.asarray(x, dtype=float)
        y = np.asarray(y, dtype=float)
        
        dx = np.diff(x)
        dy = np.diff(y)
        seg_lengths = np.sqrt(dx**2 + dy**2)
        arc = np.concatenate([[0], np.cumsum(seg_lengths)])
        total_length = arc[-1]
    
        # Arc-length positions of segment boundaries
        boundaries_arc = np.arange(0, total_length, step)
        if boundaries_arc[-1] != total_length:
            boundaries_arc = np.append(boundaries_arc, total_length)
    
        # Interpolate x, y at each boundary
        x_bound = np.interp(boundaries_arc, arc, x)
        y_bound = np.interp(boundaries_arc, arc, y)
    
        starts = np.column_stack([x_bound[:-1], y_bound[:-1]])
        ends   = np.column_stack([x_bound[1:],  y_bound[1:]])
        lengths = np.sqrt(np.sum((ends - starts)**2, axis=1))
        
        centres = (starts + ends) // 2
        dx_s, dy_s = (ends - starts).T
        theta = np.arctan2(dy_s, dx_s)
        
        return lengths, centres, theta

    def trail_model(self, coords, b, Phi, L, sigma, theta, x0, y0):
        x, y = coords
        Q = (x - x0)*np.cos(theta) + (y - y0)*np.sin(theta)
        Qperp = -(x - x0)*np.sin(theta) + (y - y0)*np.cos(theta)
        return b + (Phi / L) * (1/(2*sigma*np.sqrt(2*np.pi))) * \
               np.exp(-Qperp**2 / (2*sigma**2)) * \
               (erf((Q + L/2)/(sigma*np.sqrt(2))) - erf((Q - L/2)/(sigma*np.sqrt(2))))


    # PSF STUFF        
    def _trail_kernel_epsf(self, xg, yg, x0, y0, L, theta, epsf, n_sub=60):
        """
        Line-convolved (Veres-style) trail kernel using the local ePSF as the
        cross-track profile, in place of an assumed Gaussian. Integrates the
        ePSF along the trail direction over length L, evaluated at grid (xg, yg).
        Returns a unit-integral kernel.
        """
        s_vals = np.linspace(-L / 2, L / 2, n_sub)
        ds = s_vals[1] - s_vals[0]
        cos_t, sin_t = np.cos(theta), np.sin(theta)
    
        kernel = np.zeros(xg.shape, dtype=float)
        for s in s_vals:
            xc = x0 + s * cos_t
            yc = y0 + s * sin_t
            kernel += epsf.evaluate(xg, yg, flux=1.0, x_0=xc, y_0=yc)
        kernel *= ds
        kernel /= kernel.sum()  # renormalize over the sampled grid
        return kernel
    def psf_photometry_on_streaks(self, lengths, centres, theta, stellar_calib_baseline,
                                   epsf=None, cutout_pad=9.0):
        epsf = epsf or getattr(self, 'epsf', None)
        if epsf is None:
            raise ValueError("self.epsf not built — call build_epsf_model() first.")
    
        flux_arcsec_streak = []
        flux_net_list = []
        flux_err_list = []
        fit_ok_list = []
        flux_net_streak = 0
    
        arcmin_per_pixel = 7
    
        for l, c, t in zip(lengths, centres, theta):
            x0, y0 = c
            half = int(l / 2 + cutout_pad)
            xg, yg = np.meshgrid(
                np.arange(x0 - half, x0 + half),
                np.arange(y0 - half, y0 + half)
            )
            cutout = map_coordinates(self.fits_data, [yg.ravel(), xg.ravel()], order=1)
            cutout = cutout.reshape(xg.shape)
    
            try:
                # local field-appropriate ePSF, if you're using the fiducial grid
                kernel_img = self._trail_kernel_epsf(xg, yg, x0, y0, l, t, epsf)
    
                # linear least squares for (Phi, b) -- closed form, no curve_fit needed
                A = np.vstack([kernel_img.ravel(), np.ones(cutout.size)]).T
                coeffs, _, _, _ = np.linalg.lstsq(A, cutout.ravel(), rcond=None)
                Phi_fit, b_fit = coeffs
    
                model_at_fit = Phi_fit * kernel_img + b_fit
                resid = cutout - model_at_fit
                dof = cutout.size - 2
                chi2_reduced = np.sum(resid ** 2) / dof
    
                # analytic covariance for the linear fit -> flux uncertainty
                sigma2 = np.sum(resid ** 2) / dof
                cov = sigma2 * np.linalg.inv(A.T @ A)
                flux_err = np.sqrt(cov[0, 0])
    
                fit_ok = True
            except Exception:
                Phi_fit, flux_err, chi2_reduced = np.nan, np.nan, np.inf
                fit_ok = False
    
            flux_net = Phi_fit
            # NOTE: normalization below needs rethinking -- see comment after the function
            flux_arcsec = flux_net / (l / arcmin_per_pixel) if fit_ok else np.nan
    
            flux_net_streak += flux_net if fit_ok else 0
            flux_arcsec_streak.append(flux_arcsec)
            flux_net_list.append(flux_net)
            flux_err_list.append(flux_err)
            fit_ok_list.append(fit_ok and chi2_reduced < 3.0)
    
        flux_arcsec = np.nanmedian(flux_arcsec_streak)
        mag_arcsec = -2.5 * np.log10(flux_arcsec) + stellar_calib_baseline
        mag_total = -2.5 * np.log10(flux_net_streak) + stellar_calib_baseline
    
        flux_net_arr = np.asarray(flux_net_list, dtype=float)
        fit_ok_arr = np.asarray(fit_ok_list, dtype=bool)
    
        hampel_mask = np.ones(len(flux_net_arr), dtype=bool)
        if fit_ok_arr.sum() >= 5:
            hampel_mask[fit_ok_arr] = self.hampel_with_persistence(
                flux_net_arr[fit_ok_arr], window=10, n_sigmas=3, min_run=4
            )
        final_mask = fit_ok_arr & hampel_mask
    
        return mag_arcsec, mag_total, flux_net_list, flux_err_list, final_mask
        

        
    def ransac_streak_spline_fitting(self, points, order=2, residual_threshold=2.0,
                                  min_samples=None, max_trials=1000):
        if min_samples is None:
            min_samples = order + 2  # need enough points to constrain the poly
            
        model_class = make_poly_model(order)
        model, inliers = ransac(
            points, model_class,
            min_samples=min_samples,
            residual_threshold=residual_threshold,
            max_trials=max_trials
        )
        return inliers, model


        
    def _bilinear_sample(self, data, xs, ys):
        """Sample `data` at fractional pixel coordinates (xs, ys) via bilinear interpolation."""
        # map_coordinates expects [row, col] order, i.e. [y, x]
        coords = np.array([ys, xs])
        return map_coordinates(data, coords, order=1, mode='constant', cval=np.nan)
        
    def fit_section_profile(self, c, t, beta_2d, half_width=15, n_samples=7, sample_spacing=1.0):
        """
        Perpendicular Moffat fit for one section, centred at c with tangent angle t.
        Co-adds several parallel cross-cuts sampled along the tangent direction
        (median-stacked, with per-offset scatter used as fit weights) to improve
        SNR while staying local enough to avoid smearing across streak curvature.
    
        beta is held fixed (not fit) — the Moffat wing-shape parameter is poorly
        constrained by a single local section even at good S/N, since it's set by
        the low-flux wings specifically. Determine beta once from a Moffat fit to
        bright, isolated calibration stars and pass it in here, rather than
        re-fitting it per section.
    
        Returns corrected centre, equivalent Gaussian sigma (derived from the
        fitted Moffat FWHM, so downstream code that expects a sigma is unaffected),
        amplitude, mu offset, fit-success flag, and mu uncertainty.
        """
        p_lsf = beta_2d - 0.5
        if p_lsf <= 0.5:
            raise ValueError(f"beta_2d={beta_2d:.2f} gives a non-normalisable LSF.")
        px, py = -np.sin(t), np.cos(t)   # perpendicular direction
        tx, ty = np.cos(t), np.sin(t)    # tangent direction
    
        offsets = np.arange(-half_width, half_width + 1)
    
        # sample multiple parallel cuts along the tangent, centred on c
        along_offsets = (np.arange(n_samples) - n_samples // 2) * sample_spacing
        profiles = []
        for a in along_offsets:
            cx, cy = c[0] + a * tx, c[1] + a * ty
            xs = cx + offsets * px
            ys = cy + offsets * py
            profiles.append(self._bilinear_sample(self.fits_data, xs, ys))
        profiles = np.array(profiles)
    
        profile = np.nanmedian(profiles, axis=0)
        profile_scatter = np.nanstd(profiles, axis=0)
        # avoid zero-weight offsets (e.g. n_samples=1, or all cuts agreeing exactly)
        valid_scatter = profile_scatter[profile_scatter > 0]
        fallback_scatter = np.nanmedian(valid_scatter) if valid_scatter.size else 1.0
        profile_scatter[profile_scatter == 0] = fallback_scatter
    
        good = np.isfinite(profile) & np.isfinite(profile_scatter)
    
        A0 = np.nanmax(profile) - np.nanmedian(profile)
        B0 = np.nanmedian(profile)
        alpha0 = (half_width / 4) / np.sqrt(2 ** (1 / p_lsf) - 1)  # match Gaussian-equiv width
        p0 = [A0, 0.0, alpha0, B0]
    
        def _moffat(y, A, mu, alpha, B):
            return A * (1 + ((y - mu) / alpha) ** 2) ** (-p_lsf) + B
    
        try:
            if good.sum() < 4:  # not enough points to fit 4 params
                raise RuntimeError("too few valid samples")
    
            popt, pcov = curve_fit(
                _moffat,
                offsets[good],
                profile[good],
                p0=p0,
                sigma=profile_scatter[good],
                absolute_sigma=True,
                bounds=(
                    [-np.inf, -half_width, 0.1, -np.inf],
                    [np.inf, half_width, half_width, np.inf],
                ),
            )
            A_fit, mu_fit, alpha_fit, bkg_fit = popt
            alpha_fit = abs(alpha_fit)
    
            # convert Moffat alpha -> equivalent Gaussian sigma via matched FWHM
            fwhm = 2 * alpha_fit * np.sqrt(2 ** (1 / p_lsf) - 1)
            sigma_fit = fwhm / (2 * np.sqrt(2 * np.log(2)))
    
            # propagate alpha uncertainty into the derived sigma
            alpha_err = np.sqrt(pcov[2, 2])
            sigma_err = alpha_err * (sigma_fit / alpha_fit) if alpha_fit > 0 else np.nan
    
            mu_err = np.sqrt(pcov[1, 1])
            ok = True
    
        except (RuntimeError, ValueError):
            mu_fit, sigma_fit, A_fit, bkg_fit, mu_err = 0.0, half_width / 4, np.nan, np.nan, 0.0
            alpha_fit = np.nan
            ok = False
            
    
        # shift the section centre onto the true streak centre
        c_corrected = (c[0] + mu_fit * px, c[1] + mu_fit * py)
    
        return c_corrected, sigma_fit, A_fit, mu_fit, ok, mu_err, alpha_fit

        
    @staticmethod
    def clean_background_pixels(aperture, data, npixels=5, detect_sigma=3.0, 
                                  dilate_size=5, clip_sigma=3.0, maxiters=5):
        """
        Extract pixels from an aperture, mask detected sources, then sigma clip
        the remainder to get a robust background estimate.
        """
        # Get the pixel values and a mask marking which pixels are inside the aperture
        ap_mask = aperture.to_mask(method='center')
        cutout = ap_mask.multiply(data)
        inside = ap_mask.data > 0  # True where pixel is part of the aperture
    
        # Rough noise estimate for source detection threshold (before masking)
        _, bg_median_rough, bg_std_rough = sigma_clipped_stats(
            cutout[inside], sigma=clip_sigma, maxiters=2
        )
        threshold = bg_median_rough + detect_sigma * bg_std_rough
    
        # Detect sources within the cutout
        segm = detect_sources(cutout, threshold, npixels=npixels)
    
        if segm is not None:
            # Dilate the source mask a bit so faint wings get excluded too
            footprint = circular_footprint(radius=dilate_size)
            source_mask = segm.make_source_mask(footprint=footprint)
        else:
            source_mask = np.zeros(cutout.shape, dtype=bool)
    
        # Keep only pixels that are inside the aperture AND not flagged as a source
        good = inside & ~source_mask
        good_pixels = cutout[good]
    
        if good_pixels.size == 0:
            # Fallback: nothing survived masking, use unmasked clipped stats
            good_pixels = cutout[inside]
    
        mean, median, std = sigma_clipped_stats(good_pixels, sigma=clip_sigma, maxiters=maxiters)
        n_used = good_pixels.size
    
        return median, std, n_used, source_mask.sum()
        
    
    def photometry_of_streak(self, lengths, centres, theta, stellar_calib_baseline, ratio, zp_error, beta_lsf=None):
        """
        Takes in segments of streak to construct the photometry readings required.
        """
        flux_arcsec_streak = []
        flux_net_list = []
        area_list = []
        area = 0.0

        mu_fits = []
        sigma_fits = []
        A_fits = []
        local_bg_flags = []
        var_flux_net_list = []
        flux_error_list = []
        centres_fit = []
        mu_err_list = []
        
        if beta_lsf is None:
            beta_lsf = np.median(self.moffat_fits[:, 1])
        nu = 2*beta_lsf - 2
        if nu <= 0:
            raise ValueError(f"beta={beta_lsf:.2f} gives nu<=0; LSF undefined.")
    
        def lsf_halfwidth(alpha, EE):
            return alpha * student_t.ppf(0.5 + EE/2, nu) / np.sqrt(nu)
            
        t_list = [] 
        for l, c, t in zip(lengths, centres, theta):
            
            c_fit, sigma_fit, A_fit, mu_fit, fit_ok, mu_err, alpha_fit = self.fit_section_profile(c, t, beta_lsf)
            
            if not fit_ok or abs(mu_fit) > 8 or sigma_fit < 0.5 or sigma_fit > 12 or A_fit <= 0:
                continue
                
            mu_fits.append(mu_fit)
            A_fits.append(A_fit)
            sigma_fits.append(sigma_fit)
            centres_fit.append(c_fit)
            mu_err_list.append(mu_err)
            EE = 0.8              # must equal the EE used in stellar_calibrations
            
            half_w = lsf_halfwidth(alpha_fit, EE)
            width  = 2 * half_w
            
            # background strips at the 90% standoff, scaling with seeing
            inner  = 4 * alpha_fit
            bg_w   = 4 * half_w
            offset = inner + bg_w / 2
            c = c_fit
            on_streak = RectangularAperture(c, l, width, theta=t)
            
            #
            #width = 6 * sigma_fit  # Credits to KiyoAstro And (FORGOT THE NAME  D:  )
            #gap = 2           # gap between streak edge and bg strip (optional)
            #offset = width/2 + gap + width/2   # perpendicular distance from center

            # I will need to implement anti-crossover checks, not hard I dont think I just need to evaluate overlap etc.
            
            dx = -np.sin(t)
            dy =  np.cos(t)
            
            pos_plus  = (c[0] + offset*dx, c[1] + offset*dy)
            pos_minus = (c[0] - offset*dx, c[1] - offset*dy)
            
            off_streak_1 = RectangularAperture(pos_plus, l, width, theta = t)
            off_streak_2 = RectangularAperture(pos_minus, l, width, theta = t)
            
            phot_in = aperture_photometry(self.fits_data, on_streak)
            sum_in = phot_in["aperture_sum"][0]
            arcmin_per_pixel = 1.17 ## Will actually vary given the Fish-eye structure of the lens
            area_in = on_streak.area

            
            bkg1_med, bkg1_std, n1, nmasked1 = self.clean_background_pixels(off_streak_1, self.fits_data)
            bkg2_med, bkg2_std, n2, nmasked2 = self.clean_background_pixels(off_streak_2, self.fits_data)
            
            # Combine the two strips, weighting by number of surviving pixels
            n_total = n1 + n2
            bkg_median = (bkg1_med * n1 + bkg2_med * n2) / n_total
            bkg_std = np.sqrt((bkg1_std**2 * n1 + bkg2_std**2 * n2) / n_total)  # pooled variance
            
            # Optional: flag segments where a large fraction of background got masked out
            frac_masked = (nmasked1 + nmasked2) / (off_streak_1.area + off_streak_2.area)
            local_bg_flags.append(frac_masked > 0.3)  # you already have this list initialized
            
            flux_net = sum_in - bkg_median * area_in
            flux_pix = flux_net / area_in

            flux_net_vignet_correct, lb, ub = self.vignetting_response(flux_net, c[0], c[1])
            
            flux_error = self.flux_error(off_streak_1=off_streak_1, 
                                         off_streak_2=off_streak_2, 
                                         source_flux=flux_net, 
                                         n_pix_source=on_streak.area)

            flux_error_vignet_corrected, lb, ub  = self.vignetting_response(flux_error, c[0], c[1])
            
            flux_arcsec = flux_pix / arcmin_per_pixel
            flux_arcsec_streak.append(flux_arcsec)
            flux_net_list.append(flux_net_vignet_correct)
            area += area_in
            area_list.append(area_in)
            flux_error_list.append(flux_error_vignet_corrected)

        
        # Generate masks to eliminate unwanted readings, these could be due to a number of issues:
        # Background stars shifting the centre line
        # Stars affecting flux 
        # 
        mu_arr = np.asarray(mu_fits)
        A_arr = np.asarray(A_fits)
        sigma_arr = np.asarray(sigma_fits)
        flux_arr = np.asarray(flux_net_list)
        area_arr = np.asarray(area_list)
        flux_err_arr = np.asarray(flux_error_list)
        mu_median = np.median(mu_arr)
        mu_err_arr = np.asarray(mu_err_list)
        mad = np.median(np.abs(mu_arr - mu_median)) * 1.4826
        mu_mask = np.abs(mu_arr - mu_median) < 3 * mad
        centres_arr = np.asarray(centres_fit)
        flux_mask = self.hampel_with_persistence(flux_net_list, window=10, n_sigmas=3, min_run=3)
        flux_mask = np.asarray(flux_mask)
        
        final_mask = mu_mask & flux_mask
        
        self.plotting_light_curve(flux_arr = flux_arr, flux_err = flux_err_arr, mask = final_mask, 
                             centres = centres_arr, segment_length_array = lengths, stellar_calib_baseline = stellar_calib_baseline,
                            ratio = ratio, zp_error = zp_error)
        
        flux_net_streak = flux_arr[final_mask].sum() 
        print(f"Net sum of streak = {flux_net_streak}")
        flux_120_s_normalised = flux_net_streak * (1 / ratio) 
        area = area_arr[final_mask].sum()          # <-- now consistent with the numerator
        
        flux_arcsec = flux_net_streak / area
        mag_arcsec = -2.5*np.log10(flux_arcsec) + stellar_calib_baseline
        mag_total = -2.5*np.log10(flux_net_streak) + stellar_calib_baseline
        mag_total_120_s_normalised = -2.5*np.log10(flux_120_s_normalised) + stellar_calib_baseline
        
        snr_arr = flux_arr / flux_err_arr
        
        x_axis = np.arange(len(mu_arr))
        return mag_arcsec, mag_total, flux_net_list, mag_total_120_s_normalised


    def hampel_with_persistence(self, y, window, n_sigmas, min_run):
        """
        Flags a point as an outlier only if it deviates AND is not
        corroborated by neighboring points also deviating in the same direction.
        """
        y = np.asarray(y, dtype=float)
        n = len(y)
        candidate = np.zeros(n, dtype=bool)
        k = 1.4826
    
        # Step 1: standard Hampel candidate flagging
        for i in range(n):
            lo, hi = max(0, i - window), min(n, i + window + 1)
            local = y[lo:hi]
            med = np.median(local)
            mad = np.median(np.abs(local - med))
            if mad == 0:
                continue
            if np.abs(y[i] - med) > n_sigmas * k * mad:
                candidate[i] = True
    
        # Step 2: require persistence - a "real" deviation shows up in
        # a run of >= min_run consecutive candidates moving the same direction
        mask = np.ones(n, dtype=bool)  # True = keep, False = reject
        i = 0
        while i < n:
            if not candidate[i]:
                i += 1
                continue
            # find the run of consecutive candidate points
            j = i
            while j < n and candidate[j]:
                j += 1
            run_len = j - i
            if run_len < min_run:
                # isolated spike(s) -> genuine outlier(s), reject
                mask[i:j] = False
            # else: sustained deviation -> treat as real signal, keep it
            i = j
    
        return mask 

        

    def local_background_rms(self, fits_data, aperture, sigma_clip_sigma=3.0, maxiters=10):
        """
        Sigma-clipped RMS and Gaussianity check from pixels within a background aperture,
        local to a specific streak segment. This is the primary noise reference used in
        the CCD equation for that segment.
        """
        # 'center' (not 'exact') for noise stats: exact gives flux-weighted partial
        # pixels, which biases std for anything but pure flux summation.
        mask = aperture.to_mask(method='center')
        cutout = mask.multiply(fits_data)
        weights = mask.data
        pixels = cutout[weights > 0]
        pixels = pixels[np.isfinite(pixels)]  # drop NaN/Inf from bad-pixel masks
    
        if pixels.size < 20:
            return np.nan, np.nan, False

        sigma_clip = SigmaClip(sigma=sigma_clip_sigma, maxiters=maxiters)
        clipped = sigma_clip(pixels)
        clean = clipped.compressed()
    
        sigma_std = np.std(clean, ddof=1)
        mad = np.median(np.abs(clean - np.median(clean))) * 1.4826
        
        gaussian_ok = abs(sigma_std - mad) / sigma_std < 0.25 if sigma_std > 0 else False
    
        return sigma_std, mad, gaussian_ok
    def flux_error(self, off_streak_1, off_streak_2, source_flux, n_pix_source):
        
        rms_1, mad_1, ok_1 = self.local_background_rms(self.fits_data, off_streak_1)
        rms_2, mad_2, ok_2 = self.local_background_rms(self.fits_data, off_streak_2)

        sigma_sky = np.sqrt((rms_1**2 + rms_2**2) / 2)

        bkg_term = n_pix_source * sigma_sky**2

        egain = self.header['EGAIN']
        
        shot_term = max(source_flux, 0.0) / egain

        sigma_total = np.sqrt(bkg_term + shot_term)

        return sigma_total


        
    def computing_ratio_of_lengths(self, start_point, end_point, satellite_nid):
        level_data = np.ones(1)
        time_data = [jd2Date(self.exposure_start_jd)] 
        
        _, ra0, dec0, _ = xyToRaDecPP(time_data, [start_point[0]], [start_point[1]], level_data, self.pp)
        _, ra1, dec1, _ = xyToRaDecPP(time_data, [end_point[0]], [end_point[1]], level_data, self.pp)

        S0 = SkyCoord(ra = ra0 * u.deg, dec = dec0 * u.deg)
        S1 = SkyCoord(ra = ra1 * u.deg, dec = dec1 * u.deg)
        exposure_jd = 120 / 86400  # 0.00138889
        
        params = {'catalog': satellite_nid,'latitude': self.lat,'longitude': self.lon, 'elevation': self.alt,'startjd': self.exposure_start_jd,'stopjd': self.exposure_start_jd + exposure_jd,'stepjd': exposure_jd}   # step = full range → only 2 points returned}
        
        r = requests.get('https://satchecker.cps.iau.org/ephemeris/catalog-number-jdstep/', params=params)
        
        data = r.json()['data']
        fields = r.json()['fields']
        p0, p1 = data  # start-of-exposure, end-of-exposure records
        
        ra_i = fields.index('right_ascension_deg')
        dec_i = fields.index('declination_deg')
        
        P0 = SkyCoord(ra=p0[ra_i] * u.deg, dec=p0[dec_i] * u.deg)
        P1 = SkyCoord(ra=p1[ra_i] * u.deg, dec=p1[dec_i] * u.deg)
        length_1 = P0.separation(P1)   # returns an Angle object
        length_2 = S0.separation(S1)
        
        ratio = length_2 / length_1  # 
        return ratio


    def segment_temporal_lengths(self, centres, ratio):
        streak_duration = 120 * np.absolute(ratio)
        
        level_data = np.ones(len(centres))
        time_data = [jd2Date(self.exposure_start_jd)] * len(centres)

        # as done before 
        xs = [c[0] for c in centres]
        ys = [c[1] for c in centres]
        _, ra, dec, _ = xyToRaDecPP(time_data, xs, ys, level_data, self.pp)
        altaz_frame = AltAz(obstime=self.exposure_start, location=self.location)     
                
        coords = SkyCoord(ra=ra * u.deg, dec=dec * u.deg)
        streak_altaz = coords.transform_to(altaz_frame)
        altitude_segment = streak_altaz.alt.deg
        #streak_zenith_angle = 90 - altitude_segment[0]
        
        segment_arcs = coords[:-1].separation(coords[1:])
        segment_arcs = np.concatenate([[0 * u.deg], segment_arcs])
        cumulative_arcs = np.cumsum(segment_arcs)

        total_measured_arc = cumulative_arcs[-1]
        
        time_at_centres = (cumulative_arcs / total_measured_arc) * streak_duration

        # per-segment dwell time = local time spacing around each point
        dt_segment = np.gradient(time_at_centres.value)  # seconds, same length as segments
    
        return dt_segment, time_at_centres.value, altitude_segment
        
    def plotting_light_curve(self, flux_arr, flux_err, mask, 
                         centres, segment_length_array, stellar_calib_baseline,
                         ratio, zp_error):
    
        number_of_segments = len(flux_arr)
        dt_seg, t_axis, altitudes = self.segment_temporal_lengths(centres, ratio)
        mag_array = np.full(number_of_segments, np.nan)
        mag_err_array = np.full(number_of_segments, np.nan)
        valid_flux = flux_arr > 0
        segment_magnitude_correction = 2.5 * np.log10(dt_seg[valid_flux] / 120)
        mag_array[valid_flux] = (-2.5 * np.log10(flux_arr[valid_flux])+ stellar_calib_baseline+ segment_magnitude_correction)
        
        mag_err_array[valid_flux] = (2.5 / np.log(10)) * (flux_err[valid_flux] / flux_arr[valid_flux])
        
        accepted = mask & valid_flux
        rejected = ~mask & valid_flux
        non_positive = ~valid_flux
            
        fig, ax = plt.subplots(figsize=(7, 5), dpi=500)
        
        ax.errorbar(t_axis[accepted], mag_array[accepted], yerr=mag_err_array[accepted],
                    fmt='o', color='#003E74', ecolor='#002147', alpha=0.8,
                    markersize=4, capsize=2, label='Accepted')
    
        if np.any(rejected):
            ax.scatter(t_axis[rejected], mag_array[rejected], marker='x', color='red',
                       s=60, zorder=5, label='Rejected (quality mask)')
    
        if np.any(non_positive):
            ymin = np.nanmin(mag_array[accepted]) if np.any(accepted) else 0
            ax.scatter(t_axis[non_positive], np.full(non_positive.sum(), ymin - 0.5),
                       marker='x', color='red', s=60)
        
        ax.invert_yaxis()
        ax.set_xlabel('Time (s)', fontsize = 20)
        ax.set_ylabel('Apparent magnitude', fontsize = 20)
        ax.set_title('Satellite brightness variation along streak', fontsize = 20)
        ax.tick_params(labelsize=20)
        ax.legend(fontsize=7, loc='best')
        ax.grid(True, alpha=0.3)
        ax.yaxis.set_major_formatter(FormatStrFormatter('%.1f'))
        # text-only legend entry for the zeropoint systematic (no visible marker/line)
        handles, labels = ax.get_legend_handles_labels()
        zp_handle = Line2D([0], [0], color='none',
                            label=f'Zeropoint systematic: ±{zp_error:.3f} mag')
        handles.append(zp_handle)
        ax.legend(handles=handles)
    
        plt.tight_layout()
        plt.savefig("save_name.png", dpi = 'figure')
        plt.show()

    def computing_ratio_of_lengths_simulated(self, start_point, end_point,
                                       ra_truth_start, dec_truth_start,
                                       ra_truth_end, dec_truth_end):
        level_data = np.ones(1)
        time_data = [jd2Date(self.exposure_start_jd)]
    
        _, ra0, dec0, _ = xyToRaDecPP(time_data, [start_point[0]], [start_point[1]], level_data, self.pp)
        _, ra1, dec1, _ = xyToRaDecPP(time_data, [end_point[0]], [end_point[1]], level_data, self.pp)
    
        S0 = SkyCoord(ra=ra0 * u.deg, dec=dec0 * u.deg)
        S1 = SkyCoord(ra=ra1 * u.deg, dec=dec1 * u.deg)
    
        P0 = SkyCoord(ra=ra_truth_start * u.deg, dec=dec_truth_start * u.deg)
        P1 = SkyCoord(ra=ra_truth_end * u.deg, dec=dec_truth_end * u.deg)
    
        length_1 = P0.separation(P1).deg   # -> plain float/array, no unit
        length_2 = S0.separation(S1).deg
    
        length_1 = float(np.atleast_1d(length_1)[0])
        length_2 = float(np.atleast_1d(length_2)[0])
    
        ratio = length_2 / length_1
        return ratio, length_1, length_2
    def _compute_aperture_overlap(self, masks):
        """
        masks: list of photutils ApertureMask objects (from aperture.to_mask()).
        Returns the fraction of total on-streak aperture area that is double-counted
        (summed into more than one segment), plus the coverage array for inspection.
        """
        x0 = min(m.bbox.ixmin for m in masks)
        x1 = max(m.bbox.ixmax for m in masks)
        y0 = min(m.bbox.iymin for m in masks)
        y1 = max(m.bbox.iymax for m in masks)
    
        coverage = np.zeros((y1 - y0, x1 - x0), dtype=np.float32)
        for m in masks:
            bbox = m.bbox
            sub = coverage[bbox.iymin - y0: bbox.iymax - y0, bbox.ixmin - x0: bbox.ixmax - x0]
            sub += m.data  # fractional (0-1) weights; 'center' method gives ~binary 0/1
    
        nominal_area = sum(m.data.sum() for m in masks)
        overlap_area = np.sum(np.clip(coverage - 1, 0, None))
        overlap_fraction = overlap_area / nominal_area if nominal_area > 0 else 0.0
        return overlap_fraction, coverage
        
    def photometry_of_streak_simulated(self, lengths, centres, theta, ratio, location, check_overlap=True):
        self.stellar_calibrations(location=location)
    
        flux_net_list = []
        flux_error_list = []
        mu_fits, sigma_fits, A_fits, centres_fit, mu_err_list = [], [], [], [], []
        on_streak_masks = []
    
        def lsf_halfwidth(alpha, EE):
            return alpha * student_t.ppf(0.5 + EE/2, NU) / np.sqrt(NU)
    
        for l, c, t in zip(lengths, centres, theta):
            c_fit, sigma_fit, A_fit, mu_fit, fit_ok, mu_err, alpha_fit = self.fit_section_profile(c, t)
            print(alpha_fit)
            if not fit_ok or abs(mu_fit) > 8 or sigma_fit < 0.5 or sigma_fit > 12 or A_fit <= 0:
                continue
    
            mu_fits.append(mu_fit); A_fits.append(A_fit); sigma_fits.append(sigma_fit)
            centres_fit.append(c_fit); mu_err_list.append(mu_err)
    
            BETA_LSF = 3.5
            NU = 2 * BETA_LSF - 2
            EE = 0.99
            half_w = lsf_halfwidth(alpha_fit, EE)
            width = 2 * half_w
            inner = lsf_halfwidth(alpha_fit, 0.99)
            offset = inner + width / 2
            c = c_fit
    
            on_streak = RectangularAperture(c, l, width, theta=t)
            dx, dy = -np.sin(t), np.cos(t)
            off_streak_1 = RectangularAperture((c[0] + offset*dx, c[1] + offset*dy), l, width, theta=t)
            off_streak_2 = RectangularAperture((c[0] - offset*dx, c[1] - offset*dy), l, width, theta=t)
    
            if check_overlap:
                on_streak_masks.append(on_streak.to_mask(method='center'))
    
            phot_in = aperture_photometry(self.fits_data, on_streak)
            sum_in = phot_in["aperture_sum"][0]
            area_in = on_streak.area
    
            bkg1_med, bkg1_std, n1, nmasked1 = self.clean_background_pixels(off_streak_1, self.fits_data)
            bkg2_med, bkg2_std, n2, nmasked2 = self.clean_background_pixels(off_streak_2, self.fits_data)
            n_total = n1 + n2
            bkg_median = (bkg1_med * n1 + bkg2_med * n2) / n_total
    
            flux_net = sum_in - bkg_median * area_in
            flux_net_vignet_correct, lb, ub = self.vignetting_response(flux_net, c[0], c[1])
    
            flux_error = self.flux_error(off_streak_1, off_streak_2, flux_net, on_streak.area)
            flux_error_vignet_corrected, lb, ub = self.vignetting_response(flux_error, c[0], c[1])
    
            flux_net_list.append(flux_net_vignet_correct)
            flux_error_list.append(flux_error_vignet_corrected)
    
        flux_arr = np.asarray(flux_net_list)
        flux_err_arr = np.asarray(flux_error_list)
        centres_arr = np.asarray(centres_fit)
    
        mu_arr = np.asarray(mu_fits)
        mu_median = np.median(mu_arr)
        mad = np.median(np.abs(mu_arr - mu_median)) * 1.4826
        mu_mask = np.abs(mu_arr - mu_median) < 3 * mad
        flux_mask = np.asarray(self.hampel_with_persistence(flux_net_list, window=10, n_sigmas=3, min_run=3))
        final_mask = mu_mask & flux_mask
    
        dt_seg, t_axis, altitudes = self.segment_temporal_lengths(centres_arr, ratio)
        exposure_time = getattr(self, "exposure_time", 120)
    
        flux_dwell_corrected = np.full(len(flux_arr), np.nan)
        valid = flux_arr > 0
        flux_dwell_corrected[valid] = flux_arr[valid] * (exposure_time / dt_seg[valid])
    
        flux_total = flux_arr[final_mask].sum()
        flux_total_dwell_corrected = np.median(flux_dwell_corrected[final_mask & valid])
    
        overlap_fraction = None
        if check_overlap and on_streak_masks:
            overlap_fraction, coverage = self._compute_aperture_overlap(on_streak_masks)
            print(f"Aperture overlap: {overlap_fraction*100:.2f}% of total on-streak area double-counted")
    
        return {
            "flux_per_section": flux_arr,
            "flux_err_per_section": flux_err_arr,
            "flux_dwell_corrected_per_section": flux_dwell_corrected,
            "mask": final_mask,
            "flux_total": flux_total,
            "flux_total_dwell_corrected": flux_total_dwell_corrected,
            "aperture_overlap_fraction": overlap_fraction,
        }
    def analyse_simulated_streak(self, results, ra_truth_start, dec_truth_start,
                                  ra_truth_end, dec_truth_end, original_data=None):
        """
        Stripped-down counterpart to plot_streaks() for a synthetic frame with a
        single known-injected streak. No SatChecker involvement anywhere --
        the truth RA/Dec you pass in stands in for the satellite match.
        """
        if original_data is None:
            original_data = self.fits_data
    
        for i, r in enumerate(results):
            refined = self.refine_streak_centerline(
                *r["original_frame_xy"],
                perp_halfwidth=10,
            )
            x_seg, y_seg = refined['x_refined'], refined['y_refined']
    
            cx, cy = self.get_centerline(x_seg, y_seg, n_bins=40)
            tck, u = splprep([cx, cy], s=len(cx) * 10, k=2)
    
            u_check = np.linspace(0, 1, 500)
            spline_xs, spline_ys = splev(u_check, tck)
    
            x_mid, y_mid = splev(0.5, tck)
            _, ra_c, dec_c, _ = xyToRaDecPP(
                [jd2Date(self.exposure_start_jd)], [x_mid], [y_mid], [1], self.pp
            )
    
            # zeropoint calibration -- unchanged, uses field stars only, no satellite involved
            
            # ground-truth-based ratio, replacing the SatChecker ephemeris lookup
            start_point = (spline_xs[0], spline_ys[0])
            end_point = (spline_xs[-1], spline_ys[-1])
            ratio, length_true, length_measured = self.computing_ratio_of_lengths_simulated(
                start_point, end_point,
                ra_truth_start, dec_truth_start, ra_truth_end, dec_truth_end
            )
            print(f"Streak {i}: true length={length_true:.4f} deg, "
                  f"measured length={length_measured:.4f} deg, ratio={ratio:.4f}")
    
            lengths, centres, theta = self.split_streak_into_sections(spline_xs, spline_ys)
    
            lengths, centres, theta = self.split_streak_into_sections(spline_xs, spline_ys)

            result = self.photometry_of_streak_simulated(
                lengths=lengths, centres=centres, theta=theta,
                ratio=ratio, location=(ra_c, dec_c)
            )
            print(result)
    
            # simple 2-panel diagnostic instead of the 3-panel (no satellite-match panel)
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))
            vmin, vmax = np.percentile(self.fits_data, [1, 99.5])
    
            ax1.imshow(self.fits_data, origin='upper', cmap='gray', vmin=vmin, vmax=vmax)
            ax1.plot(x_seg, y_seg, color="lime", linewidth=1.5, label="Hough/refined")
            ax1.set_xlim(np.min(x_seg) - 100, np.max(x_seg) + 100)
            ax1.set_ylim(np.min(y_seg) - 100, np.max(y_seg) + 100)
            ax1.set_title("Refined centerline")
    
            ax2.imshow(self.fits_data, origin='upper', cmap='gray', vmin=vmin, vmax=vmax)
            ax2.plot(spline_xs, spline_ys, color="blue", linewidth=3, label="Fitted spline")
            ax2.set_xlim(np.min(x_seg) - 100, np.max(x_seg) + 100)
            ax2.set_ylim(np.min(y_seg) - 100, np.max(y_seg) + 100)
            ax2.set_title(f"Spline fit vs. injected truth (ratio={ratio:.3f})")
    
            plt.tight_layout()
            plt.savefig(f"streak_simulated_{i}.png")
            plt.show()


In [28]:
directory = "C:/Users/carlo/Summer Project/260713-14"
files = ["demo_moffat3.fits"]

for file in files:
    #file_dir = os.path.join(directory, file)
    test = OperatingFitsFiles(file, site = 'paranal', simulated = True) ## For some reason beyond my comprehension I have hardcoded the site, calling self.site breaks Satchecker
    test.opening_fits_file()
    #test.build_epsf_model()
    test.run_crop_procedure()
    test.undistort_fits()
    test.run_hough_transform()

SIMPLE  =                    T / conforms to FITS standard                      BITPIX  =                  -32 / array data type                                NAXIS   =                    2 / number of array dimensions                     NAXIS1  =                 8750                                                  NAXIS2  =                 8750                                                  EXPTIME =   120.00000000000000 /Exposure time in seconds                        EXPOSURE=   120.00000000000000 /Exposure time in seconds                        CCD-TEMP=   7.068634033203E-02 /CCD temperature at start of exposure in C       XPIXSZ  =   3.7599999999999998 /Pixel Width in microns (after binning)          YPIXSZ  =   3.7599999999999998 /Pixel Height in microns (after binning)         XBINNING=                    1 /Binning factor in width                         YBINNING=                    1 /Binning factor in height                        XORGSUBF=                    0 / Subfram

KeyboardInterrupt: 